# 🚀 GPU Fundamentals to Advanced: NVIDIA CUDA, ML & RAPIDS
### A Complete Hands-On Training Notebook for the NVIDIA A100

**Author:** Manoj Kumar | **Target Hardware:** NVIDIA A100 (40/80 GB HBM2e) | **Date:** 2026

---

## 📚 Curriculum Overview

| Module | Topic | Concepts |
|--------|-------|----------|
| 0 | **Setup & GPU Inventory** | Environment, nvidia-smi, device properties |
| 1 | **GPU Architecture** | SMs, warps, thread hierarchy, memory hierarchy |
| 2 | **CuPy — GPU NumPy** | Arrays, kernels, FFT, benchmarks |
| 3 | **Numba CUDA Kernels** | @cuda.jit, indexing, shared memory, coalescing |
| 4 | **Memory Optimization** | Pinned memory, UM, pools, roofline model |
| 5 | **RAPIDS cuDF** | GPU DataFrames, ETL, cudf.pandas |
| 6 | **RAPIDS cuML** | GPU-accelerated scikit-learn equivalents |
| 7 | **RAPIDS cuGraph** | PageRank, BFS, community detection |
| 8 | **Dask-cuDF** | Multi-GPU, out-of-core, partitioned DataFrames |
| 9 | **PyTorch GPU Deep Dive** | Streams, async transfers, torch.compile |
| 10 | **Tensor Cores & AMP** | TF32, BF16, autocast, GradScaler, checkpointing |
| 11 | **Profiling & Tuning** | torch.profiler, NVTX, roofline validation |
| 12 | **Capstone Pipeline** | End-to-end GPU-accelerated ML workflow |

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:
- **Explain** how GPU hardware executes parallel programs (warps, SMs, memory hierarchy)
- **Write** CUDA kernels in Python using Numba and CuPy
- **Identify** memory bottlenecks and apply coalescing, pinned memory, and pooling
- **Use** the full RAPIDS ecosystem for GPU-accelerated data science
- **Train** deep learning models with mixed precision on A100 Tensor Cores
- **Profile** and **tune** GPU code with `torch.profiler` and the roofline model

---

## 🖥️ Hardware: NVIDIA A100 Fast Facts

| Spec | Value |
|------|-------|
| Architecture | Ampere |
| CUDA Cores | 6,912 |
| Tensor Cores | 432 (3rd Gen) |
| SMs | 108 |
| HBM2e Memory | 40 GB or 80 GB |
| Memory Bandwidth | 1,555–2,000 GB/s |
| FP32 Peak | ~19.5 TFLOPS |
| TF32 Peak | ~156 TFLOPS |
| BF16 / FP16 TC Peak | ~312 TFLOPS |
| Interconnect | NVLink 3.0 / PCIe 4.0 |

> 💡 **Tip:** Run every cell in order. Each module builds on the previous one.

# ═══════════════════════════════════════════════════════════════
# MODULE 0 — Setup & GPU Inventory
# ═══════════════════════════════════════════════════════════════

## What You'll Learn
- Install all required libraries for the full curriculum
- Inspect your A100 hardware programmatically
- Run your first GPU benchmark to see the speedup

## Key Terms
- **CUDA**: NVIDIA's parallel computing platform and programming model
- **VRAM / HBM2e**: Video RAM — the high-bandwidth memory on your GPU
- **SM (Streaming Multiprocessor)**: The core execution unit on an NVIDIA GPU
- **Compute Capability**: A version number (e.g. 8.0 for A100) encoding what GPU features are available

In [ ]:
# ── MODULE 0 · Cell 1 ── Install all dependencies ──────────────────────────
# Run once; restart kernel afterwards if first install
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

print("Installing core PyTorch & ML libraries...")
pip("torch>=2.2.0", "torchvision", "torchaudio", "--index-url",
    "https://download.pytorch.org/whl/cu121")

print("Installing CuPy (CUDA 12.x build)...")
pip("cupy-cuda12x")

print("Installing Numba...")
pip("numba")

print("Installing RAPIDS (cuDF, cuML, cuGraph, Dask-cuDF)...")
pip(
    "--extra-index-url", "https://pypi.nvidia.com",
    "cudf-cu12", "cuml-cu12", "cugraph-cu12",
    "dask-cudf-cu12", "cucim", "raft-dask-cu12",
)

print("Installing supporting libraries...")
pip(
    "matplotlib", "seaborn", "pandas>=2.0",
    "numpy", "scipy", "scikit-learn",
    "networkx",  # CPU graph baseline for cuGraph comparison
    "tqdm", "rich",
)

print("\n✅ All dependencies installed. Restart the kernel now if this was a fresh install.")

In [ ]:
# ── MODULE 0 · Cell 2 ── Global imports & configuration ────────────────────
import os, time, math, warnings, gc
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn

# ── Seaborn style ──────────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
NVIDIA_GREEN = "#76b900"

# ── Device detection ───────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Primary compute device : {DEVICE}")

if torch.cuda.is_available():
    print(f"CUDA version           : {torch.version.cuda}")
    print(f"cuDNN version          : {torch.backends.cudnn.version()}")
    print(f"Number of GPUs visible : {torch.cuda.device_count()}")
else:
    print("⚠️  No CUDA GPU detected — code will fall back to CPU where possible.")

In [ ]:
# ── MODULE 0 · Cell 3 ── GPU Inventory ────────────────────────────────────
import subprocess

def nvidia_smi():
    """Run nvidia-smi and print output."""
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        print(result.stdout)
    except FileNotFoundError:
        print("nvidia-smi not found — are NVIDIA drivers installed?")

nvidia_smi()

In [ ]:
# ── MODULE 0 · Cell 4 ── Detailed GPU Properties (PyTorch) ─────────────────
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"\n{'='*60}")
        print(f"  GPU {i}: {p.name}")
        print(f"{'='*60}")
        print(f"  Compute Capability      : {p.major}.{p.minor}")
        print(f"  Total VRAM              : {p.total_memory / 1e9:.2f} GB")
        print(f"  Streaming Multiprocessors (SMs) : {p.multi_processor_count}")
        print(f"  Max threads per SM      : {p.max_threads_per_multi_processor}")
        print(f"  Max threads per block   : {p.max_threads_per_block}")
        print(f"  Warp size               : {p.warp_size}")
        print(f"  Max shared mem / block  : {p.max_shared_memory_per_block / 1024:.0f} KB")
        print(f"  L2 Cache size           : {p.l2_cache_size / 1e6:.0f} MB")
        print(f"  Clock Rate (GHz)        : {p.clock_rate / 1e6:.2f}")
        print(f"  Memory Clock Rate (GHz) : {p.memory_clock_rate / 1e6:.2f}")
        print(f"  Memory Bus Width (bits) : {p.memory_bus_width}")
        # Theoretical memory bandwidth
        bw = (p.memory_bus_width / 8) * (p.memory_clock_rate * 1e3) * 2 / 1e9
        print(f"  Theoretical Mem BW      : {bw:.0f} GB/s")
        # Estimated FP32 peak
        # CUDA cores ≈ SMs × 128 for Ampere
        cuda_cores = p.multi_processor_count * 128
        fp32_tflops = cuda_cores * 2 * (p.clock_rate * 1e3) / 1e12
        print(f"  Est. CUDA Cores         : {cuda_cores}")
        print(f"  Est. FP32 Peak          : {fp32_tflops:.1f} TFLOPS")

    # A100-specific notes
    print("\n💡 A100 Ampere specifics:")
    print("   - Compute Capability 8.0")
    print("   - TF32 Tensor Cores: ~156 TFLOPS (default for torch.matmul)")
    print("   - BF16/FP16 Tensor Cores: ~312 TFLOPS")
    print("   - INT8 Tensor Cores: ~624 TOPS")
    print("   - MIG (Multi-Instance GPU): up to 7 isolated GPU instances")
    print("   - NVLink 3.0: 600 GB/s bidirectional (A100 SXM pairing)")
else:
    print("No GPU — properties unavailable.")

In [ ]:
# ── MODULE 0 · Cell 5 ── First Benchmark: CPU vs GPU Matrix Multiply ────────
# This is your "Hello, GPU!" — a simple sanity check showing WHY we use GPUs.

def benchmark_matmul(size=4096, dtype=torch.float32, n_warmup=3, n_runs=10):
    results = {}

    # ── CPU ──────────────────────────────────────────────────────────────────
    A_cpu = torch.randn(size, size, dtype=dtype)
    B_cpu = torch.randn(size, size, dtype=dtype)
    for _ in range(n_warmup):
        _ = torch.mm(A_cpu, B_cpu)
    t0 = time.perf_counter()
    for _ in range(n_runs):
        _ = torch.mm(A_cpu, B_cpu)
    cpu_ms = (time.perf_counter() - t0) / n_runs * 1000
    results["CPU (FP32)"] = cpu_ms

    if torch.cuda.is_available():
        # ── GPU FP32 ──────────────────────────────────────────────────────────
        A_gpu = A_cpu.cuda()
        B_gpu = B_cpu.cuda()
        start_evt, end_evt = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        # Warm-up
        for _ in range(n_warmup):
            torch.mm(A_gpu, B_gpu)
        torch.cuda.synchronize()
        start_evt.record(); [torch.mm(A_gpu, B_gpu) for _ in range(n_runs)]; end_evt.record()
        torch.cuda.synchronize()
        results["GPU FP32"] = start_evt.elapsed_time(end_evt) / n_runs

        # ── GPU TF32 (A100 default) ────────────────────────────────────────────
        torch.backends.cuda.matmul.allow_tf32 = True
        for _ in range(n_warmup):
            torch.mm(A_gpu, B_gpu)
        torch.cuda.synchronize()
        start_evt.record(); [torch.mm(A_gpu, B_gpu) for _ in range(n_runs)]; end_evt.record()
        torch.cuda.synchronize()
        results["GPU TF32 (A100)"] = start_evt.elapsed_time(end_evt) / n_runs

        # ── GPU BF16 Tensor Core ───────────────────────────────────────────────
        A_bf16 = A_gpu.to(torch.bfloat16)
        B_bf16 = B_gpu.to(torch.bfloat16)
        torch.backends.cuda.matmul.allow_tf32 = True
        for _ in range(n_warmup):
            torch.mm(A_bf16, B_bf16)
        torch.cuda.synchronize()
        start_evt.record(); [torch.mm(A_bf16, B_bf16) for _ in range(n_runs)]; end_evt.record()
        torch.cuda.synchronize()
        results["GPU BF16 TC"] = start_evt.elapsed_time(end_evt) / n_runs

    print(f"\n{'─'*45}")
    print(f"  Matrix Multiply Benchmark  ({size}×{size})")
    print(f"{'─'*45}")
    for label, ms in results.items():
        # FLOPS for matmul = 2 * N^3
        flops = 2 * size**3
        tflops = flops / (ms / 1000) / 1e12
        speedup = results["CPU (FP32)"] / ms
        print(f"  {label:<22}: {ms:7.2f} ms | {tflops:6.1f} TFLOPS | {speedup:5.1f}× vs CPU")
    print(f"{'─'*45}")
    return results

results = benchmark_matmul(size=4096)

# ── Visualization ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
labels = list(results.keys())
times = list(results.values())
colors = [NVIDIA_GREEN if "GPU" in l else "#e74c3c" for l in labels]
bars = ax.barh(labels, times, color=colors, edgecolor="white", height=0.5)
ax.set_xlabel("Latency (ms) — lower is better")
ax.set_title(f"Matrix Multiply Benchmark (4096×4096)\nCPU vs NVIDIA A100 GPU", fontsize=13, fontweight="bold")
for bar, t in zip(bars, times):
    ax.text(t + max(times)*0.01, bar.get_y() + bar.get_height()/2,
            f"{t:.2f} ms", va="center", fontsize=10)
ax.set_xlim(0, max(times) * 1.25)
plt.tight_layout()
plt.savefig("m0_matmul_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved as m0_matmul_benchmark.png")

# ═══════════════════════════════════════════════════════════════
# MODULE 1 — GPU Architecture: The Mental Model
# ═══════════════════════════════════════════════════════════════

## Why GPUs are Fast (and When They're Not)

The fundamental difference between a CPU and a GPU:

| Property | CPU (e.g. Intel Xeon) | GPU (NVIDIA A100) |
|----------|----------------------|-------------------|
| Core count | 8–128 large cores | 6,912 small CUDA cores |
| Design goal | **Minimize latency** (1 task, fast) | **Maximize throughput** (millions of tasks) |
| Cache | Very large (L3 ~32 MB) | Smaller per core, but HBM is fast |
| Clock speed | 3–5 GHz | 1.3–1.8 GHz |
| Best for | Sequential logic, OS, branchy code | Dense math, parallelizable algorithms |

**Key insight:** CPUs use big caches and complex out-of-order execution to make *one* task go fast. GPUs use thousands of simple cores running in lockstep to make *millions* of tasks go simultaneously.

---

## The A100 Hardware Hierarchy

```
GPU (A100)
├── 108 Streaming Multiprocessors (SMs)
│   ├── 4 × Warp Schedulers                 ← issue 1 warp/cycle each
│   ├── 64 CUDA FP32 cores per SM partition
│   ├── 4 × 3rd-gen Tensor Core (TC)
│   ├── 192 KB L1 cache + shared memory (configurable)
│   └── Registers (256 KB per SM)
├── 40 MB L2 Cache (shared)
└── 80 GB HBM2e (High Bandwidth Memory, ~2 TB/s)
```

## Thread Hierarchy

```
Grid  (your kernel launch)
└── Blocks  (up to 1024 threads, run on one SM)
    └── Warps  (32 threads, execute in lockstep)
        └── Threads  (one CUDA core per thread)
```

**Critical rule:** All threads in a *warp* execute the **same instruction** at the same time.  
If half the warp takes `if` and half takes `else`, both branches run sequentially → **warp divergence** → 2× slower.

In [ ]:
# ── MODULE 1 · Cell 1 ── Architecture Diagram: CPU vs GPU ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle("CPU vs GPU Architecture", fontsize=16, fontweight="bold")

# ── CPU diagram ───────────────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("CPU (e.g. Intel Xeon)\n'Latency-Optimized'", fontsize=13, color="#2c3e50")

def draw_box(ax, x, y, w, h, label, color, fontsize=9):
    rect = plt.Rectangle((x, y), w, h, fc=color, ec="white", lw=1.5, zorder=2)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha="center", va="center",
            fontsize=fontsize, color="white", fontweight="bold", zorder=3)

# Control units (big) & ALUs
for i in range(4):
    row, col = divmod(i, 2)
    ox, oy = col*4.5 + 0.3, row*4.5 + 1.5
    draw_box(ax, ox,      oy+2.5, 3.8, 1.5, "Control Unit", "#8e44ad", fontsize=8)
    draw_box(ax, ox,      oy+1.0, 1.8, 1.3, "ALU", "#2980b9")
    draw_box(ax, ox+2.0,  oy+1.0, 1.8, 1.3, "ALU", "#2980b9")
    draw_box(ax, ox,      oy,     3.8, 0.9, f"L1 Cache (core {i})", "#16a085", fontsize=7)

draw_box(ax, 0.3, 0.2, 9.4, 0.7, "Shared L3 Cache (32 MB)", "#e67e22")
draw_box(ax, 0.3, 0.0, 9.4, 0.25, "DRAM  (~50 GB/s)", "#c0392b", fontsize=7)
ax.text(5, 9.7, "4 big latency-optimized cores", ha="center", fontsize=10, style="italic")

# ── GPU diagram ───────────────────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("GPU (NVIDIA A100)\n'Throughput-Optimized'", fontsize=13, color="#2c3e50")

# Draw 108 SMs as tiny squares (show a representative 36)
colors_sm = [NVIDIA_GREEN]*36
cols_sm = 9
for idx in range(36):
    r, c = divmod(idx, cols_sm)
    x = c * 1.05 + 0.15
    y = 9.0 - r * 0.92
    rect = plt.Rectangle((x, y-0.75), 0.9, 0.78, fc=NVIDIA_GREEN, ec="white", lw=0.5)
    ax.add_patch(rect)
    if idx == 0:
        ax.text(x+0.45, y-0.37, "SM", ha="center", va="center", fontsize=6, color="white", fontweight="bold")

ax.text(5, 5.7, "... 108 SMs total (showing 36) ...", ha="center",
        fontsize=9, style="italic", color="#555")
draw_box(ax, 0.15, 4.9, 9.7, 0.65, "L2 Cache (40 MB)", "#e67e22")
draw_box(ax, 0.15, 4.0, 9.7, 0.80, "HBM2e  (~2,000 GB/s — 40× CPU!", "#c0392b")
ax.text(5, 3.7,  "Each SM has 64 FP32 cores + Tensor Cores + 192 KB shared mem",
        ha="center", fontsize=8.5, style="italic", color="#444")
ax.text(5, 3.3,  "6,912 CUDA cores execute millions of lightweight threads",
        ha="center", fontsize=8.5, style="italic", color="#444")

plt.tight_layout()
plt.savefig("m1_cpu_vs_gpu.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── MODULE 1 · Cell 2 ── Thread Hierarchy Visualization ────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14); ax.set_ylim(0, 6.5); ax.axis("off")
ax.set_title("CUDA Thread Hierarchy: Grid → Block → Warp → Thread", fontsize=14, fontweight="bold")

# Grid outline
grid_rect = plt.Rectangle((0.2, 0.2), 13.6, 6.0, fc="#ecf0f1", ec="#bdc3c7", lw=2)
ax.add_patch(grid_rect)
ax.text(7, 6.0, "GRID  (your entire kernel launch)", ha="center", fontsize=11,
        fontweight="bold", color="#2c3e50")

# 4 blocks
block_colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12"]
block_labels = ["Block (0,0)", "Block (1,0)", "Block (0,1)", "Block (1,1)"]
bx_offsets = [0.5, 7.1, 0.5, 7.1]
by_offsets = [3.2, 3.2, 0.4, 0.4]

for bi in range(4):
    bx, by = bx_offsets[bi], by_offsets[bi]
    bw, bh = 6.3, 2.5
    block_rect = plt.Rectangle((bx, by), bw, bh, fc=block_colors[bi], alpha=0.15,
                                ec=block_colors[bi], lw=2)
    ax.add_patch(block_rect)
    ax.text(bx + bw/2, by + bh - 0.22, block_labels[bi], ha="center", fontsize=9,
            fontweight="bold", color=block_colors[bi])

    # 2 warps per block (each warp = 32 threads, show 8 threads as squares)
    for wi in range(2):
        wy = by + 0.3 + wi * 1.0
        ax.text(bx + 0.15, wy + 0.55, f"Warp {wi}\n(32 threads)", ha="left",
                va="center", fontsize=7, color="#555")
        for ti in range(8):  # show 8 of the 32
            tx = bx + 1.4 + ti * 0.58
            ty = wy + 0.2
            t_rect = plt.Rectangle((tx, ty), 0.46, 0.6,
                                    fc=block_colors[bi], alpha=0.7, ec="white", lw=0.8)
            ax.add_patch(t_rect)
            lbl = f"T{wi*8+ti}" if ti < 7 else "…"
            ax.text(tx+0.23, ty+0.3, lbl, ha="center", va="center",
                    fontsize=6, color="white", fontweight="bold")

ax.text(7, 0.05, "All threads in a warp execute the SAME instruction simultaneously (SIMT)",
        ha="center", fontsize=10, style="italic", color="#c0392b")

plt.tight_layout()
plt.savefig("m1_thread_hierarchy.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── MODULE 1 · Cell 3 ── Memory Hierarchy Pyramid ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis("off")
ax.set_title("A100 GPU Memory Hierarchy Pyramid", fontsize=14, fontweight="bold")

levels = [
    # (y_base, height, width_half, label, sublabel, color)
    (0.2, 1.0, 4.8, "HBM2e (80 GB)",      "~2,000 GB/s  |  ~350 ns latency",  "#c0392b"),
    (1.4, 1.0, 4.0, "L2 Cache (40 MB)",   "~200 GB/s  (shared across all SMs)","#e67e22"),
    (2.6, 1.0, 3.2, "L1 / Shared Mem\n(192 KB / SM)", "≥10 TB/s (on-chip)  |  ~30 cycles", "#f1c40f"),
    (3.8, 1.0, 2.4, "Registers\n(256 KB / SM)",        "Fastest — zero-cycle access",          NVIDIA_GREEN),
]

for y, h, hw, lbl, sub, col in levels:
    # Trapezoid-like via polygon
    xl = 5 - hw;  xr = 5 + hw
    xl_top = 5 - (hw - 0.6);  xr_top = 5 + (hw - 0.6)
    poly = plt.Polygon([[xl, y], [xr, y], [xr_top, y+h], [xl_top, y+h]],
                        fc=col, ec="white", lw=1.5, alpha=0.85)
    ax.add_patch(poly)
    ax.text(5, y + h/2 + 0.05, lbl, ha="center", va="center",
            fontsize=10, fontweight="bold", color="white")
    ax.text(5, y + 0.12, sub, ha="center", va="bottom",
            fontsize=7.5, color="white", style="italic")

# Arrows
for y_a in [1.25, 2.45, 3.65]:
    ax.annotate("", xy=(5, y_a+0.1), xytext=(5, y_a),
                arrowprops=dict(arrowstyle="<->", color="#333", lw=1.5))

annotations = [
    (8.5, 0.7,  "Largest, slowest\nAll data starts here"),
    (8.5, 1.9,  "Shared by all SMs\nGood for reuse across blocks"),
    (8.5, 3.1,  "Programmer-managed!\nUse for intra-block reuse"),
    (8.5, 4.3,  "Compiler-managed\nLocal variables land here"),
]
for ax_x, ax_y, note in annotations:
    ax.text(ax_x, ax_y, note, ha="center", va="center",
            fontsize=8, color="#2c3e50",
            bbox=dict(boxstyle="round,pad=0.3", fc="#ecf0f1", ec="#bdc3c7"))
    ax.annotate("", xy=(5 + [4.8,4.0,3.2,2.4][annotations.index((ax_x,ax_y,note))]*0.5 + 0.1,
                         [0.7,1.9,3.1,4.3][annotations.index((ax_x,ax_y,note))]),
                xytext=(ax_x - 1.3, ax_y),
                arrowprops=dict(arrowstyle="-", color="#aaa", lw=1, linestyle="dashed"))

ax.text(5, 5.3,
        "Rule of thumb: keep hot data in registers → shared memory → L2 → minimize HBM accesses",
        ha="center", fontsize=9, style="italic", color="#8e44ad",
        bbox=dict(boxstyle="round,pad=0.4", fc="#f5eef8", ec="#8e44ad"))

plt.tight_layout()
plt.savefig("m1_memory_hierarchy.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── MODULE 1 · Cell 4 ── Warp Divergence Demonstration ─────────────────────
# We demonstrate divergence using Numba CUDA kernels.
# Import Numba here; we will use it heavily in Module 3.

from numba import cuda
import numpy as np

N = 1 << 20  # 1M elements

# ── Without divergence: all threads do same work ───────────────────────────
@cuda.jit
def no_divergence_kernel(x, out):
    i = cuda.grid(1)
    if i < x.size:
        # Same math for every thread — no divergence
        out[i] = x[i] * 2.0 + 1.0

# ── With divergence: even/odd threads take different branches ───────────────
@cuda.jit
def with_divergence_kernel(x, out):
    i = cuda.grid(1)
    if i < x.size:
        if i % 2 == 0:          # ← half the warp goes here
            out[i] = x[i] * 2.0 + 1.0
        else:                    # ← other half goes here → SERIALIZED
            out[i] = x[i] * 3.0 - 1.0

if torch.cuda.is_available():
    x_np = np.random.rand(N).astype(np.float32)
    x_d  = cuda.to_device(x_np)
    out_d = cuda.device_array(N, dtype=np.float32)

    threads_per_block = 256
    blocks = (N + threads_per_block - 1) // threads_per_block

    # ── Warm-up ───────────────────────────────────────────────────────────
    no_divergence_kernel[blocks, threads_per_block](x_d, out_d)
    with_divergence_kernel[blocks, threads_per_block](x_d, out_d)
    cuda.synchronize()

    # ── Timed runs ────────────────────────────────────────────────────────
    REPS = 200

    t0 = time.perf_counter()
    for _ in range(REPS):
        no_divergence_kernel[blocks, threads_per_block](x_d, out_d)
    cuda.synchronize()
    t_no_div = (time.perf_counter() - t0) / REPS * 1000

    t0 = time.perf_counter()
    for _ in range(REPS):
        with_divergence_kernel[blocks, threads_per_block](x_d, out_d)
    cuda.synchronize()
    t_div = (time.perf_counter() - t0) / REPS * 1000

    print(f"\n{'─'*50}")
    print(f"  Warp Divergence Benchmark  (N = {N:,})")
    print(f"{'─'*50}")
    print(f"  No divergence   : {t_no_div:.3f} ms")
    print(f"  With divergence : {t_div:.3f} ms   ({t_div/t_no_div:.2f}× slower)")
    print(f"{'─'*50}")
    print("\n💡 Takeaway: alternating branches force the warp to run both paths sequentially.")
    print("   Avoid fine-grained if/else inside kernels whose condition depends on thread index.")
else:
    print("No GPU — skipping warp divergence benchmark.")

# ═══════════════════════════════════════════════════════════════
# MODULE 2 — CuPy: GPU-Accelerated NumPy
# ═══════════════════════════════════════════════════════════════

## What is CuPy?

**CuPy** is a drop-in GPU replacement for NumPy and SciPy. Almost every `numpy.XXX` operation has a `cupy.XXX` equivalent that runs on the GPU — you just change the import.

```python
# CPU (NumPy)
import numpy as np
x = np.array([1, 2, 3])
y = np.fft.fft(x)

# GPU (CuPy) — same API!
import cupy as cp
x = cp.array([1, 2, 3])
y = cp.fft.fft(x)
```

## Why CuPy?

- **Zero CUDA expertise required** for most operations
- Supports: array creation, math, linalg, FFT, random, sparse, custom kernels
- Interoperates with NumPy, PyTorch, and RAPIDS seamlessly
- For large arrays (≥ 1M elements), typically **10–100× faster** than NumPy

## Key Concepts

| Concept | Description |
|---------|-------------|
| `cp.array()` | Create a GPU ndarray (data lives in HBM2e) |
| `cp.asnumpy()` / `.get()` | Copy GPU → CPU (expensive!) |
| `cp.cuda.Event` | High-resolution GPU timer |
| `cp.ElementwiseKernel` | Write custom GPU element-wise ops in CUDA C |
| `cp.RawKernel` | Execute raw CUDA C code |
| Memory pool | CuPy reuses GPU memory to avoid costly `cudaMalloc` |

In [ ]:
# ── MODULE 2 · Cell 1 ── CuPy Basics: Arrays & Device Transfers ─────────────
try:
    import cupy as cp
    HAS_CUPY = True
    print(f"CuPy version : {cp.__version__}")
    print(f"CUDA version : {cp.cuda.runtime.runtimeGetVersion()}")
    print(f"GPU device   : {cp.cuda.Device(0).attributes}")
except ImportError:
    HAS_CUPY = False
    print("⚠️  CuPy not installed. Install with: pip install cupy-cuda12x")
    print("   All CuPy cells will be skipped.")

if HAS_CUPY:
    # ── Creating arrays ────────────────────────────────────────────────────
    a_cpu = np.random.rand(1_000_000).astype(np.float32)

    # Transfer to GPU
    t0 = time.perf_counter()
    a_gpu = cp.asarray(a_cpu)            # host → device copy
    cp.cuda.Stream.null.synchronize()
    h2d_ms = (time.perf_counter() - t0) * 1000

    # Transfer back
    t0 = time.perf_counter()
    a_back = cp.asnumpy(a_gpu)           # device → host copy
    d2h_ms = (time.perf_counter() - t0) * 1000

    size_mb = a_cpu.nbytes / 1e6
    print(f"\n  Array size         : {size_mb:.1f} MB")
    print(f"  Host → Device (H2D): {h2d_ms:.2f} ms  ({size_mb/h2d_ms*1e3:.1f} MB/s)")
    print(f"  Device → Host (D2H): {d2h_ms:.2f} ms  ({size_mb/d2h_ms*1e3:.1f} MB/s)")
    print(f"\n⚠️  Transfer cost is real! A100 PCIe 4.0 peak ≈ ~32 GB/s H2D.")
    print("   Strategy: move data once, do all work on GPU, transfer only final results.")

    # ── Array properties ───────────────────────────────────────────────────
    b = cp.zeros((4, 4), dtype=cp.float16)
    print(f"\n  b.device : {b.device}")
    print(f"  b.dtype  : {b.dtype}")
    print(f"  b.shape  : {b.shape}")
    print(f"  b.nbytes : {b.nbytes} bytes")

    # ── get_array_module: write device-agnostic code ───────────────────────
    def relu(x):
        """Works on both CPU (numpy) and GPU (cupy) arrays."""
        xp = cp.get_array_module(x)
        return xp.maximum(x, 0)

    print(f"\n  relu(np array) type : {type(relu(np.array([-1, 2, -3])))}")
    print(f"  relu(cp array) type : {type(relu(cp.array([-1, 2, -3])))}")

In [ ]:
# ── MODULE 2 · Cell 2 ── CuPy vs NumPy Operation Benchmarks ────────────────
if HAS_CUPY:
    def cupy_time(fn, *args, warmup=5, reps=50):
        for _ in range(warmup):
            fn(*args)
        cp.cuda.Stream.null.synchronize()
        start = cp.cuda.Event(); end = cp.cuda.Event()
        start.record()
        for _ in range(reps):
            fn(*args)
        end.record()
        end.synchronize()
        return start.elapsed_time(end) / reps

    def numpy_time(fn, *args, reps=20):
        # Warm-up
        for _ in range(3): fn(*args)
        t0 = time.perf_counter()
        for _ in range(reps): fn(*args)
        return (time.perf_counter() - t0) / reps * 1000

    N  = 10_000_000
    size = (N,)
    size_2d = (4096, 4096)

    a_np = np.random.rand(*size).astype(np.float32)
    b_np = np.random.rand(*size).astype(np.float32)
    a_cp = cp.asarray(a_np)
    b_cp = cp.asarray(b_np)

    A_np = np.random.rand(*size_2d).astype(np.float32)
    B_np = np.random.rand(*size_2d).astype(np.float32)
    A_cp = cp.asarray(A_np)
    B_cp = cp.asarray(B_np)

    benchmarks = {}

    # Element-wise multiply
    benchmarks["Elementwise Mul\n(10M floats)"] = (
        numpy_time(lambda: a_np * b_np),
        cupy_time(lambda: a_cp * b_cp),
    )
    # Sum reduction
    benchmarks["Sum Reduction\n(10M floats)"] = (
        numpy_time(lambda: a_np.sum()),
        cupy_time(lambda: a_cp.sum()),
    )
    # Sorting
    benchmarks["Sort\n(10M floats)"] = (
        numpy_time(lambda: np.sort(a_np)),
        cupy_time(lambda: cp.sort(a_cp)),
    )
    # FFT
    benchmarks["FFT\n(10M floats)"] = (
        numpy_time(lambda: np.fft.fft(a_np)),
        cupy_time(lambda: cp.fft.fft(a_cp)),
    )
    # Matrix multiply
    benchmarks["MatMul\n(4096×4096)"] = (
        numpy_time(lambda: A_np @ B_np, reps=5),
        cupy_time(lambda: A_cp @ B_cp),
    )

    # ── Print table ────────────────────────────────────────────────────────
    print(f"\n{'─'*62}")
    print(f"  {'Operation':<26} {'NumPy (ms)':>12} {'CuPy (ms)':>12} {'Speedup':>8}")
    print(f"{'─'*62}")
    for op, (np_t, cp_t) in benchmarks.items():
        op_short = op.replace("\n", " ")
        print(f"  {op_short:<26} {np_t:>12.3f} {cp_t:>12.3f} {np_t/cp_t:>7.1f}×")
    print(f"{'─'*62}")

    # ── Bar chart ─────────────────────────────────────────────────────────
    ops = [k.replace("\n", "\n") for k in benchmarks]
    np_times = [v[0] for v in benchmarks.values()]
    cp_times = [v[1] for v in benchmarks.values()]
    speedups = [n/c for n, c in zip(np_times, cp_times)]

    x = np.arange(len(ops))
    width = 0.35
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.bar(x - width/2, np_times, width, label="NumPy (CPU)", color="#e74c3c", edgecolor="white")
    ax1.bar(x + width/2, cp_times, width, label="CuPy  (A100)", color=NVIDIA_GREEN, edgecolor="white")
    ax1.set_xticks(x); ax1.set_xticklabels(ops, fontsize=8)
    ax1.set_ylabel("Latency (ms) — lower is better")
    ax1.set_title("NumPy vs CuPy: Absolute Latency", fontweight="bold")
    ax1.legend()

    ax2.bar(x, speedups, color=NVIDIA_GREEN, edgecolor="white")
    ax2.set_xticks(x); ax2.set_xticklabels(ops, fontsize=8)
    ax2.set_ylabel("Speedup (×) — higher is better")
    ax2.set_title("CuPy Speedup over NumPy (A100)", fontweight="bold")
    ax2.axhline(1, color="red", linestyle="--", linewidth=1, label="Breakeven (1×)")
    for xi, s in zip(x, speedups):
        ax2.text(xi, s + 0.3, f"{s:.1f}×", ha="center", fontsize=9, fontweight="bold")
    ax2.legend()

    plt.tight_layout()
    plt.savefig("m2_cupy_benchmark.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("CuPy not available — skipping benchmark.")

In [ ]:
# ── MODULE 2 · Cell 3 ── Custom CuPy Kernels (ElementwiseKernel & RawKernel)
if HAS_CUPY:
    # ── ElementwiseKernel: fused SELU activation ──────────────────────────
    # Writing this as a fused kernel avoids two separate passes over memory.
    selu_kernel = cp.ElementwiseKernel(
        in_params='float32 x',
        out_params='float32 y',
        operation='''
            const float alpha = 1.6732632423543772f;
            const float scale = 1.0507009873554805f;
            y = (x >= 0) ? scale * x : scale * alpha * (expf(x) - 1.0f);
        ''',
        name='selu'
    )

    x_test = cp.array([-2.0, -1.0, 0.0, 1.0, 2.0], dtype=cp.float32)
    print("Custom SELU kernel output:")
    print("  Input  :", cp.asnumpy(x_test))
    print("  Output :", cp.asnumpy(selu_kernel(x_test)))

    # ── ReductionKernel: custom L2 norm ───────────────────────────────────
    l2_norm = cp.ReductionKernel(
        in_params='float32 x',
        out_params='float32 y',
        map_expr='x * x',
        reduce_expr='a + b',
        post_map_expr='y = sqrtf(a)',
        identity='0',
        name='l2_norm'
    )
    v = cp.array([3.0, 4.0], dtype=cp.float32)
    print(f"\n  L2 norm([3, 4]) = {float(l2_norm(v)):.4f}  (expected 5.0)")

    # ── RawKernel: fully custom CUDA C string ─────────────────────────────
    raw_add_kernel = cp.RawKernel(r'''
    extern "C" __global__ void vec_add_scaled(
        const float* a, const float* b, float* c, float scale, int n)
    {
        int i = blockDim.x * blockIdx.x + threadIdx.x;
        if (i < n) {
            c[i] = (a[i] + b[i]) * scale;
        }
    }
    ''', 'vec_add_scaled')

    N_raw = 1_000_000
    a_r = cp.random.rand(N_raw, dtype=cp.float32)
    b_r = cp.random.rand(N_raw, dtype=cp.float32)
    c_r = cp.empty(N_raw, dtype=cp.float32)

    threads = 256
    blocks  = (N_raw + threads - 1) // threads
    raw_add_kernel((blocks,), (threads,), (a_r, b_r, c_r, cp.float32(2.5), N_raw))
    cp.cuda.Stream.null.synchronize()

    # Verify
    expected = (a_r + b_r) * 2.5
    print(f"\n  RawKernel correctness check: max error = {float(cp.max(cp.abs(c_r - expected))):.2e}")
    print("\n💡 Use ElementwiseKernel for fused ops, RawKernel for full CUDA C control.")
    print("   Both avoid separate Python-level kernel launches → better GPU utilization.")
else:
    print("CuPy not available.")

# ═══════════════════════════════════════════════════════════════
# MODULE 3 — Writing CUDA Kernels with Numba
# ═══════════════════════════════════════════════════════════════

## Why Numba?

Numba lets you write **CUDA kernels in pure Python** with a decorator. No compilation step, no C code, no Makefiles — just Python functions decorated with `@cuda.jit`.

The kernel runs on the GPU, and Numba compiles your Python code to PTX (GPU assembly) the first time it's called.

## Step 1: Thread Indexing Formula

Every CUDA thread needs to know **which element it is responsible for**. This is the most important formula in GPU programming:

```
i = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
```

| Variable | Meaning |
|----------|---------|
| `cuda.threadIdx.x` | Thread index *within* a block (0 to blockDim-1) |
| `cuda.blockIdx.x` | Block index *within* the grid (0 to gridDim-1) |
| `cuda.blockDim.x` | Number of threads per block (you choose, typically 256) |

```
Grid (N elements, e.g. 1024):
 Block 0 (threads 0–255)  Block 1 (threads 256–511)  Block 2  ...  Block 3
 [T0 T1 ... T255]          [T256 T257 ... T511]         ...
  i = 0*256 + 0 = 0         i = 1*256 + 0 = 256
  i = 0*256 + 1 = 1         i = 1*256 + 1 = 257
  ...                        ...
```

## Step 2: Coalesced Memory Access

GPUs have a 128-byte cache line. If thread 0 reads element 0, thread 1 reads element 1, etc. → **one memory transaction** for 32 threads. This is **coalesced** access.

If thread 0 reads element 0, thread 1 reads element 32, etc. → **32 memory transactions**. This is **strided** access and can be 10× slower.

**Rule:** threads in a warp should access **consecutive** memory addresses.

## Step 3: Shared Memory

Shared memory is like an L1 cache you control explicitly. It's on-chip (fast) and shared by all threads in a block.

Use it when:
- Multiple threads need the **same input data** (avoids re-reading from slow HBM)
- You want to **tile** a large computation (e.g. matrix multiply)

```python
sh = cuda.shared.array(shape=(256,), dtype=numba.float32)
sh[cuda.threadIdx.x] = global_data[i]  # each thread loads its element
cuda.syncthreads()                      # ← MUST sync before any thread reads others' data
```

In [ ]:
# ── MODULE 3 · Cell 1 ── First Kernel: Vector Addition ─────────────────────
import numba
from numba import cuda, float32, int32

# ── Step 1: Write the kernel ───────────────────────────────────────────────
@cuda.jit
def vector_add(a, b, c):
    """Each thread computes one element of c = a + b."""
    i = cuda.grid(1)          # shorthand for blockIdx.x * blockDim.x + threadIdx.x
    if i < a.shape[0]:        # bounds check — ALWAYS do this
        c[i] = a[i] + b[i]

if torch.cuda.is_available():
    N = 1_000_000

    # ── Step 2: Allocate arrays ────────────────────────────────────────────
    a_host = np.ones(N, dtype=np.float32)
    b_host = np.ones(N, dtype=np.float32) * 2.0

    # Move to GPU
    a_dev = cuda.to_device(a_host)
    b_dev = cuda.to_device(b_host)
    c_dev = cuda.device_array(N, dtype=np.float32)

    # ── Step 3: Configure the launch grid ─────────────────────────────────
    threads_per_block = 256
    blocks_per_grid   = (N + threads_per_block - 1) // threads_per_block
    print(f"  N                = {N:,}")
    print(f"  threads_per_block= {threads_per_block}")
    print(f"  blocks_per_grid  = {blocks_per_grid:,}")
    print(f"  Total threads    = {blocks_per_grid * threads_per_block:,}  (≥ N)")

    # ── Step 4: Launch the kernel ──────────────────────────────────────────
    vector_add[blocks_per_grid, threads_per_block](a_dev, b_dev, c_dev)
    cuda.synchronize()

    # ── Step 5: Verify correctness ─────────────────────────────────────────
    c_host = c_dev.copy_to_host()
    expected = a_host + b_host
    max_error = np.max(np.abs(c_host - expected))
    print(f"\n  Correctness check : max_error = {max_error:.2e}  ✅" if max_error < 1e-5
          else f"\n  ❌ ERROR: max_error = {max_error:.2e}")
    print(f"  c[:5] = {c_host[:5]}  (expected [3. 3. 3. 3. 3.])")

    # ── Timing ────────────────────────────────────────────────────────────
    REPS = 200
    # Warm-up
    for _ in range(10):
        vector_add[blocks_per_grid, threads_per_block](a_dev, b_dev, c_dev)
    cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(REPS):
        vector_add[blocks_per_grid, threads_per_block](a_dev, b_dev, c_dev)
    cuda.synchronize()
    gpu_ms = (time.perf_counter() - t0) / REPS * 1000

    t0 = time.perf_counter()
    for _ in range(REPS):
        _ = a_host + b_host
    cpu_ms = (time.perf_counter() - t0) / REPS * 1000

    print(f"\n  CPU (NumPy) : {cpu_ms:.3f} ms")
    print(f"  GPU (Numba) : {gpu_ms:.3f} ms   ({cpu_ms/gpu_ms:.1f}× speedup)")

    # Bandwidth calculation: reads a, b (2×N×4 bytes) + writes c (1×N×4 bytes)
    bw_gb = (3 * N * 4) / (gpu_ms / 1000) / 1e9
    peak_bw = 2000  # A100 HBM2e peak GB/s
    print(f"  Achieved mem BW : {bw_gb:.0f} GB/s  ({bw_gb/peak_bw*100:.1f}% of A100 peak {peak_bw} GB/s)")
    print("\n💡 Vector add is memory-bandwidth bound (compute is trivial).")
    print("   This kernel is a good baseline for measuring your GPU's memory bandwidth.")
else:
    print("No GPU available.")

In [ ]:
# ── MODULE 3 · Cell 2 ── Coalesced vs Strided Memory Access ────────────────
# Coalesced: thread i accesses element i  → sequential → 1 cache line / 32 threads
# Strided:   thread i accesses element i*S → non-sequential → S cache lines

@cuda.jit
def coalesced_read(data, out):
    """Thread i reads data[i] — coalesced (sequential addresses in a warp)."""
    i = cuda.grid(1)
    if i < data.shape[0]:
        out[i] = data[i] * 2.0

@cuda.jit
def strided_read(data, out, stride):
    """Thread i reads data[i * stride] — strided (scattered addresses in a warp)."""
    i = cuda.grid(1)
    if i * stride < data.shape[0]:
        out[i] = data[i * stride] * 2.0

if torch.cuda.is_available():
    # Large enough to see bandwidth differences
    N = 1 << 24           # 16M floats = 64 MB
    data_d   = cuda.to_device(np.random.rand(N).astype(np.float32))
    out_d    = cuda.device_array(N // 32, dtype=np.float32)   # strided output is smaller
    out_coal = cuda.device_array(N, dtype=np.float32)

    TPB  = 256
    blks_coal   = (N + TPB - 1) // TPB
    blks_stride = (N // 32 + TPB - 1) // TPB

    REPS = 100
    results_coal = {}
    results_strd = {}

    for stride in [1, 2, 4, 8, 16, 32]:
        # Coalesced (stride 1)
        for _ in range(10): coalesced_read[blks_coal, TPB](data_d, out_coal)
        cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(REPS): coalesced_read[blks_coal, TPB](data_d, out_coal)
        cuda.synchronize()
        results_coal[stride] = (time.perf_counter() - t0) / REPS * 1000

        # Strided
        for _ in range(10): strided_read[blks_stride, TPB](data_d, out_d, int(stride))
        cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(REPS): strided_read[blks_stride, TPB](data_d, out_d, int(stride))
        cuda.synchronize()
        results_strd[stride] = (time.perf_counter() - t0) / REPS * 1000

    print(f"\n{'─'*60}")
    print(f"  {'Stride':<8} {'Coalesced (ms)':>16} {'Strided (ms)':>14} {'Penalty':>8}")
    print(f"{'─'*60}")
    for s in [1, 2, 4, 8, 16, 32]:
        penalty = results_strd[s] / results_coal[1]
        print(f"  {s:<8} {results_coal[1]:>16.3f} {results_strd[s]:>14.3f} {penalty:>7.1f}×")
    print(f"{'─'*60}")

    # Visualization
    strides = [1, 2, 4, 8, 16, 32]
    coal_times = [results_coal[1]] * len(strides)
    strd_times = [results_strd[s] for s in strides]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(strides, coal_times, "o--", color="#2ecc71", label="Coalesced (stride=1)", linewidth=2)
    ax.plot(strides, strd_times, "s-",  color="#e74c3c", label="Strided", linewidth=2, marker="s")
    ax.set_xscale("log", base=2); ax.set_xlabel("Stride"); ax.set_ylabel("Latency (ms)")
    ax.set_title("Memory Access Pattern: Coalesced vs Strided\n(A100 HBM2e, 16M float32)", fontweight="bold")
    ax.legend(); ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig("m3_coalescing.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n💡 At stride=32, each warp issues 32 separate cache line fetches instead of 1 → ~10–30× slower.")
else:
    print("No GPU available.")

In [ ]:
# ── MODULE 3 · Cell 3 ── Shared Memory: Tiled Matrix Multiply ──────────────
# Without tiling: each element of C requires K loads from A and K loads from B → 2*N^3 HBM reads
# With tiling (tile size T): we load each tile into shared mem once, reuse T times → 2*N^3/T HBM reads

from numba import float32 as nb_float32

TILE = 16  # tile size

@cuda.jit
def matmul_naive(A, B, C):
    """Naive matmul — each thread reads A[row,:] and B[:,col] directly from HBM."""
    row, col = cuda.grid(2)
    M, K = A.shape
    _, N = B.shape
    if row < M and col < N:
        s = 0.0
        for k in range(K):
            s += A[row, k] * B[k, col]
        C[row, col] = s

@cuda.jit
def matmul_tiled(A, B, C):
    """Tiled matmul — uses shared memory to reduce HBM reads by TILE×."""
    # Shared memory tiles (allocated at compile time via literal TILE)
    sA = cuda.shared.array(shape=(TILE, TILE), dtype=nb_float32)
    sB = cuda.shared.array(shape=(TILE, TILE), dtype=nb_float32)

    tx = cuda.threadIdx.x; ty = cuda.threadIdx.y
    row = cuda.blockIdx.y * TILE + ty
    col = cuda.blockIdx.x * TILE + tx
    M, K = A.shape
    _, N = B.shape

    acc = nb_float32(0.0)
    for tile_k in range((K + TILE - 1) // TILE):
        # Load one tile from A and B into shared memory
        sA[ty, tx] = A[row, tile_k * TILE + tx] if (row < M and tile_k * TILE + tx < K) else 0.0
        sB[ty, tx] = B[tile_k * TILE + ty, col] if (tile_k * TILE + ty < K and col < N) else 0.0
        cuda.syncthreads()          # ← all threads must finish loading before any can compute

        # Compute partial dot product using shared tile
        for k in range(TILE):
            acc += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()          # ← all threads must finish computing before next tile load

    if row < M and col < N:
        C[row, col] = acc

if torch.cuda.is_available():
    M = K = N = 512   # keep small so naive is tractable
    A_np = np.random.rand(M, K).astype(np.float32)
    B_np = np.random.rand(K, N).astype(np.float32)

    A_d = cuda.to_device(A_np)
    B_d = cuda.to_device(B_np)
    C_naive = cuda.device_array((M, N), dtype=np.float32)
    C_tiled = cuda.device_array((M, N), dtype=np.float32)

    grid_2d   = ((N + TILE - 1) // TILE, (M + TILE - 1) // TILE)
    block_2d  = (TILE, TILE)

    # Warm-up + run
    for _ in range(3):
        matmul_naive[grid_2d, block_2d](A_d, B_d, C_naive)
        matmul_tiled[grid_2d, block_2d](A_d, B_d, C_tiled)
    cuda.synchronize()

    REPS = 50
    t0 = time.perf_counter()
    for _ in range(REPS): matmul_naive[grid_2d, block_2d](A_d, B_d, C_naive)
    cuda.synchronize()
    t_naive = (time.perf_counter() - t0) / REPS * 1000

    t0 = time.perf_counter()
    for _ in range(REPS): matmul_tiled[grid_2d, block_2d](A_d, B_d, C_tiled)
    cuda.synchronize()
    t_tiled = (time.perf_counter() - t0) / REPS * 1000

    # Verify
    ref = A_np @ B_np
    err_naive = np.max(np.abs(C_naive.copy_to_host() - ref))
    err_tiled = np.max(np.abs(C_tiled.copy_to_host() - ref))

    print(f"\n  Matrix size: {M}×{K} × {K}×{N}")
    print(f"  Tile size  : {TILE}×{TILE}")
    print(f"  Naive   : {t_naive:.3f} ms  |  max error vs NumPy: {err_naive:.2e}")
    print(f"  Tiled   : {t_tiled:.3f} ms  |  max error vs NumPy: {err_tiled:.2e}  ({t_naive/t_tiled:.2f}× faster)")
    print(f"\n  HBM reads reduced by ~{TILE}× with tiling (TILE={TILE})")
    print("  In practice, cuBLAS/torch.mm uses tile sizes of 128+ for maximum throughput on A100.")
else:
    print("No GPU available.")

In [ ]:
# ── MODULE 3 · Cell 4 ── Atomic Operations: GPU Histogram ──────────────────
# When multiple threads write to the SAME output location, you need atomics.
# Atomic ops guarantee correctness but can serialize → use carefully.

@cuda.jit
def histogram_naive(data, bins, n_bins):
    """WRONG without atomics! Multiple threads increment same bin → race condition."""
    i = cuda.grid(1)
    if i < data.shape[0]:
        bin_idx = int(data[i] * n_bins)
        if 0 <= bin_idx < n_bins:
            bins[bin_idx] += 1          # ← RACE CONDITION (do NOT do this!)

@cuda.jit
def histogram_atomic(data, bins, n_bins):
    """CORRECT: use cuda.atomic.add to safely increment shared bins."""
    i = cuda.grid(1)
    if i < data.shape[0]:
        bin_idx = int(data[i] * n_bins)
        if 0 <= bin_idx < n_bins:
            cuda.atomic.add(bins, bin_idx, 1)   # ← atomic increment ✅

if torch.cuda.is_available():
    N_HIST  = 10_000_000
    N_BINS  = 256
    data_np = np.random.rand(N_HIST).astype(np.float32)
    data_d  = cuda.to_device(data_np)

    # Atomic histogram
    bins_d = cuda.to_device(np.zeros(N_BINS, dtype=np.int32))
    TPB_h  = 256
    BLKS_h = (N_HIST + TPB_h - 1) // TPB_h

    histogram_atomic[BLKS_h, TPB_h](data_d, bins_d, N_BINS)
    cuda.synchronize()
    bins_gpu = bins_d.copy_to_host()

    # CPU reference
    bins_cpu, _ = np.histogram(data_np, bins=N_BINS, range=(0, 1))

    max_err = np.max(np.abs(bins_gpu.astype(np.int64) - bins_cpu.astype(np.int64)))
    total_gpu = bins_gpu.sum()
    print(f"  Total elements counted (GPU): {total_gpu:,}  (expected {N_HIST:,})")
    print(f"  Max bin error vs NumPy      : {max_err}")
    print(f"  Correctness: {'✅ PASS' if max_err == 0 else '❌ FAIL'}")

    # Visualize histogram
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(np.linspace(0, 1, N_BINS), bins_gpu, width=1/N_BINS, color=NVIDIA_GREEN, alpha=0.8, label="GPU histogram")
    ax.plot(np.linspace(0, 1, N_BINS), bins_cpu, color="#e74c3c", lw=1.5, label="NumPy reference")
    ax.set_xlabel("Value"); ax.set_ylabel("Count"); ax.legend()
    ax.set_title(f"GPU Atomic Histogram (N={N_HIST:,}, {N_BINS} bins)", fontweight="bold")
    plt.tight_layout()
    plt.savefig("m3_histogram.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n💡 Atomic operations are correct but can serialize on hot bins.")
    print("   Optimization: first accumulate into shared-memory bins per block,")
    print("   then do one atomic.add per block per bin → reduces contention ~256×.")
else:
    print("No GPU available.")

# ═══════════════════════════════════════════════════════════════
# MODULE 4 — Memory Optimization Deep Dive
# ═══════════════════════════════════════════════════════════════

## The Memory Wall

For many GPU workloads, **memory bandwidth is the bottleneck**, not compute.  
The **Roofline Model** visualizes this:

```
           Peak Compute (TFLOPS)
  TFLOPS ↑ _____________________
           |                    |
           |     Compute-bound  |
           |                    |
           |_____/
           |    /  Ridge Point
           |   /
           |  / Memory-bound (slope = mem. bandwidth)
           | /
           |/________________________→ Arithmetic Intensity (FLOP/byte)
```

- **Arithmetic Intensity (AI)** = FLOPs ÷ bytes of memory traffic
- If AI < ridge point → **memory-bound** (faster memory or better reuse helps)
- If AI ≥ ridge point → **compute-bound** (more compute or better data types help)
- **A100 ridge point ≈ 9.7 FLOP/byte** (FP32: ~19.5 TFLOPS ÷ ~2000 GB/s)

## 4 Memory Strategies

| Strategy | When to Use | A100 Benefit |
|----------|-------------|--------------|
| **Pinned (page-locked) memory** | Frequent H2D/D2H transfers | 2–3× faster PCIe transfers |
| **Asynchronous transfers** | Overlap compute & transfer | Hide transfer latency |
| **Memory pools** | Many small allocations | Avoids costly `cudaMalloc` overhead |
| **Unified Memory** | Large datasets, infrequent access | Simplify code; auto paging |

In [ ]:
# ── MODULE 4 · Cell 1 ── Pinned Memory vs Pageable Memory ─────────────────
# Pageable memory (default): OS can swap it; GPU DMA must stage through OS buffer → slower
# Pinned (page-locked) memory: cannot be swapped; GPU DMA directly → faster transfers

if torch.cuda.is_available():
    sizes_mb = [64, 128, 256, 512, 1024]
    h2d_pageable, h2d_pinned = [], []

    for sz_mb in sizes_mb:
        n = sz_mb * 1024 * 1024 // 4  # number of float32

        # ── Pageable (default) ────────────────────────────────────────────
        t_page = []
        for _ in range(5):
            x_page = torch.zeros(n, dtype=torch.float32)
            t0 = time.perf_counter()
            x_page.cuda()
            torch.cuda.synchronize()
            t_page.append((time.perf_counter() - t0) * 1000)
        h2d_pageable.append(np.median(t_page))

        # ── Pinned ────────────────────────────────────────────────────────
        t_pin = []
        for _ in range(5):
            x_pin = torch.zeros(n, dtype=torch.float32).pin_memory()
            t0 = time.perf_counter()
            x_pin.cuda(non_blocking=True)      # async transfer with pinned memory!
            torch.cuda.synchronize()
            t_pin.append((time.perf_counter() - t0) * 1000)
        h2d_pinned.append(np.median(t_pin))

        del x_page, x_pin
        gc.collect(); torch.cuda.empty_cache()

    # Print table
    print(f"\n{'─'*60}")
    print(f"  {'Size (MB)':<12} {'Pageable H2D (ms)':>20} {'Pinned H2D (ms)':>18} {'Speedup':>8}")
    print(f"{'─'*60}")
    for sz, tp, tpin in zip(sizes_mb, h2d_pageable, h2d_pinned):
        bw_page = sz / (tp / 1000)
        bw_pin  = sz / (tpin / 1000)
        print(f"  {sz:<12} {tp:>18.2f} ms  {tpin:>16.2f} ms  {tp/tpin:>6.1f}×  "
              f"[{bw_pin:.0f} MB/s]")
    print(f"{'─'*60}")

    # Visualization
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sizes_mb, h2d_pageable, "o-", color="#e74c3c", label="Pageable", lw=2)
    ax.plot(sizes_mb, h2d_pinned,   "s-", color=NVIDIA_GREEN, label="Pinned (page-locked)", lw=2)
    ax.set_xlabel("Transfer size (MB)"); ax.set_ylabel("Latency (ms)")
    ax.set_title("H2D Transfer: Pageable vs Pinned Memory (PCIe 4.0)", fontweight="bold")
    ax.legend(); ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig("m4_pinned_memory.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n💡 Always use .pin_memory() + non_blocking=True in DataLoader for training.")
    print("   PyTorch DataLoader does this automatically when you set pin_memory=True.")
else:
    print("No GPU.")

In [ ]:
# ── MODULE 4 · Cell 2 ── GPU Memory Monitoring & Pools ────────────────────
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

    print("── Before any allocation ─────────────────────────────────────")
    alloc = torch.cuda.memory_allocated() / 1e6
    reserv = torch.cuda.memory_reserved() / 1e6
    print(f"  allocated  : {alloc:.1f} MB")
    print(f"  reserved   : {reserv:.1f} MB  (caching allocator pre-reserved pool)")

    x = torch.randn(1000, 1000, device="cuda")
    print("\n── After allocating 1000×1000 float32 (~4 MB) ───────────────")
    print(f"  allocated : {torch.cuda.memory_allocated()/1e6:.1f} MB")
    print(f"  reserved  : {torch.cuda.memory_reserved()/1e6:.1f} MB")

    del x
    torch.cuda.empty_cache()
    print("\n── After del + empty_cache() ────────────────────────────────")
    print(f"  allocated : {torch.cuda.memory_allocated()/1e6:.1f} MB  (freed)")
    print(f"  reserved  : {torch.cuda.memory_reserved()/1e6:.1f} MB  (cache may still hold)")

    # Simulate a training step memory profile
    print("\n── Simulating forward/backward pass memory trace ────────────")
    model = nn.Sequential(
        nn.Linear(1024, 2048), nn.ReLU(),
        nn.Linear(2048, 2048), nn.ReLU(),
        nn.Linear(2048, 512)
    ).cuda()

    batch = torch.randn(256, 1024, device="cuda")
    mem_stamps = []

    mem_stamps.append(("start", torch.cuda.memory_allocated()/1e6))
    out = model(batch)
    mem_stamps.append(("forward", torch.cuda.memory_allocated()/1e6))
    loss = out.sum()
    loss.backward()
    mem_stamps.append(("backward", torch.cuda.memory_allocated()/1e6))
    del loss, out, batch
    torch.cuda.empty_cache()
    mem_stamps.append(("cleanup", torch.cuda.memory_allocated()/1e6))

    print(f"  {'Stage':<12} {'Allocated (MB)':>16}")
    print(f"  {'─'*30}")
    for stage, mem in mem_stamps:
        print(f"  {stage:<12} {mem:>16.1f}")

    # CuPy memory pool
    if HAS_CUPY:
        pool = cp.get_default_memory_pool()
        before_mb = pool.used_bytes() / 1e6
        tmp = [cp.zeros(1024*1024, dtype=cp.float32) for _ in range(10)]
        after_alloc = pool.used_bytes() / 1e6
        del tmp
        after_del = pool.used_bytes() / 1e6
        pool.free_all_blocks()
        after_free = pool.used_bytes() / 1e6
        print(f"\n── CuPy Memory Pool ─────────────────────────────────────────")
        print(f"  Before alloc       : {before_mb:.1f} MB")
        print(f"  After alloc 40 MB  : {after_alloc:.1f} MB")
        print(f"  After del (cached) : {after_del:.1f} MB  ← cached, NOT returned to OS")
        print(f"  After free_all()   : {after_free:.1f} MB  ← fully released")
        print("  💡 CuPy reuses cached blocks for future allocs → avoids cudaMalloc overhead")
    
    del model
    torch.cuda.empty_cache()
else:
    print("No GPU.")

In [ ]:
# ── MODULE 4 · Cell 3 ── Roofline Model: A100 Analysis ─────────────────────
# The roofline model tells you whether your kernel is memory-bound or compute-bound.

# A100 specifications
A100_PEAK_FP32_TFLOPS   = 19.5
A100_PEAK_BF16TC_TFLOPS = 312.0
A100_MEM_BW_TBS         = 2.0          # TB/s HBM2e
A100_RIDGE_FP32  = A100_PEAK_FP32_TFLOPS / A100_MEM_BW_TBS    # ~9.75 FLOP/byte
A100_RIDGE_BF16  = A100_PEAK_BF16TC_TFLOPS / A100_MEM_BW_TBS  # ~156 FLOP/byte

# Representative kernel arithmetic intensities
kernels = {
    "Vector Add\n(elementwise)":        (0.25,  "memory-bound"),
    "Element-wise\nactivation (ReLU)":  (0.5,   "memory-bound"),
    "Batch Norm":                        (2.0,   "memory-bound"),
    "Dense Layer\n(small batch)":        (6.0,   "memory-bound"),
    "MatMul FP32\n(large)":             (25.0,  "compute-bound"),
    "Attention\n(FlashAttn)":            (35.0,  "compute-bound"),
    "MatMul BF16 TC\n(large)":          (180.0, "compute-bound"),
}

fig, ax = plt.subplots(figsize=(12, 6))
ai_range = np.logspace(-1, 3, 1000)

# FP32 roofline
fp32_roof = np.minimum(A100_PEAK_FP32_TFLOPS, A100_MEM_BW_TBS * ai_range)
ax.loglog(ai_range, fp32_roof, "-", color="#3498db", lw=2.5, label="FP32 Roofline")

# BF16 TC roofline
bf16_roof = np.minimum(A100_PEAK_BF16TC_TFLOPS, A100_MEM_BW_TBS * ai_range)
ax.loglog(ai_range, bf16_roof, "--", color=NVIDIA_GREEN, lw=2.5, label="BF16 Tensor Core Roofline")

# Ridge points
ax.axvline(A100_RIDGE_FP32, color="#3498db", lw=1, linestyle=":", alpha=0.6)
ax.axvline(A100_RIDGE_BF16, color=NVIDIA_GREEN, lw=1, linestyle=":", alpha=0.6)
ax.text(A100_RIDGE_FP32*1.1, 0.15, f"FP32 ridge\n{A100_RIDGE_FP32:.1f} FLOP/byte",
        fontsize=8, color="#3498db")
ax.text(A100_RIDGE_BF16*1.1, 0.15, f"BF16 ridge\n{A100_RIDGE_BF16:.0f} FLOP/byte",
        fontsize=8, color=NVIDIA_GREEN)

# Plot kernels
colors_k = {"memory-bound": "#e74c3c", "compute-bound": "#2ecc71"}
for name, (ai, region) in kernels.items():
    perf_fp32 = min(A100_PEAK_FP32_TFLOPS, A100_MEM_BW_TBS * ai)
    ax.scatter(ai, perf_fp32, s=100, color=colors_k[region], zorder=5)
    ax.annotate(name, (ai, perf_fp32), textcoords="offset points",
                xytext=(5, 5), fontsize=7.5)

legend_patches = [
    mpatches.Patch(color="#e74c3c", label="Memory-bound kernels"),
    mpatches.Patch(color="#2ecc71", label="Compute-bound kernels"),
]
ax.legend(handles=legend_patches + ax.get_lines()[:2], fontsize=9)
ax.set_xlabel("Arithmetic Intensity (FLOP / byte)", fontsize=11)
ax.set_ylabel("Performance (TFLOPS)", fontsize=11)
ax.set_title("Roofline Model: NVIDIA A100 (FP32 & BF16 Tensor Core)", fontsize=13, fontweight="bold")
ax.grid(True, which="both", alpha=0.3)
ax.set_xlim(0.1, 1000); ax.set_ylim(0.05, 1000)

plt.tight_layout()
plt.savefig("m4_roofline.png", dpi=150, bbox_inches="tight")
plt.show()

print("💡 Key takeaways:")
print("   • Most DL ops (LayerNorm, elementwise activations) are MEMORY-BOUND on A100")
print("   • Large matmuls with BF16 Tensor Cores CAN hit the compute roof → use AMP!")
print("   • To lift a kernel past the ridge: increase batch size / sequence len / tile size")
print(f"   • A100 FP32 ridge: {A100_RIDGE_FP32:.1f} FLOP/byte | BF16TC ridge: {A100_RIDGE_BF16:.0f} FLOP/byte")

# ═══════════════════════════════════════════════════════════════
# MODULE 5 — RAPIDS cuDF: GPU DataFrames
# ═══════════════════════════════════════════════════════════════

## What is cuDF?

**cuDF** is a GPU DataFrame library that mirrors the Pandas API. Data lives in GPU memory; all operations run on CUDA kernels. Switch from Pandas to cuDF by changing the import — same code, GPU speed.

```python
import pandas as pd      →   import cudf as cudf
df = pd.read_csv(...)    →   df = cudf.read_csv(...)
df.groupby(...).mean()   →   df.groupby(...).mean()   # same!
```

## Why GPU DataFrames?

| Operation | Pandas (CPU) | cuDF (GPU) | Typical Speedup |
|-----------|-------------|-----------|-----------------|
| `read_csv` (1 GB) | ~10 s | ~0.5 s | 20× |
| `groupby + agg` | slow | fast | 50–100× |
| `merge / join` | slow | fast | 30–50× |
| String operations | very slow | fast | 100×+ |
| `sort_values` | slow | fast | 20–40× |

## cudf.pandas — Zero-Code Acceleration

RAPIDS 23.x introduced `cudf.pandas`, which patches the `pandas` module itself:

```python
import cudf.pandas
cudf.pandas.install()        # monkey-patch pandas
import pandas as pd          # normal pandas import
# → now ALL pandas code runs on GPU automatically!
```

This means you can take **any existing Pandas script** and accelerate it with one extra line.

In [ ]:
# ── MODULE 5 · Cell 1 ── cuDF Setup & Basic Operations ─────────────────────
try:
    import cudf
    import cudf.pandas as cudf_pd
    HAS_CUDF = True
    print(f"cuDF version: {cudf.__version__}")
except ImportError:
    HAS_CUDF = False
    print("⚠️  cuDF not installed. Install via: pip install cudf-cu12 --extra-index-url https://pypi.nvidia.com")
    print("   All cuDF cells will be skipped.")

if HAS_CUDF:
    import pandas as pd

    # ── Create a GPU DataFrame ─────────────────────────────────────────────
    data = {
        "user_id":  np.random.randint(0, 1000, size=1_000_000),
        "product":  np.random.randint(0, 100,  size=1_000_000),
        "amount":   np.random.rand(1_000_000).astype(np.float32) * 500,
        "quantity": np.random.randint(1, 10,   size=1_000_000),
    }

    df_pd = pd.DataFrame(data)
    df_cd = cudf.DataFrame(data)   # lives in GPU HBM2e

    print(f"\n  Pandas DataFrame info:")
    print(f"    shape   : {df_pd.shape}")
    print(f"    memory  : {df_pd.memory_usage(deep=True).sum() / 1e6:.1f} MB  (CPU RAM)")
    print(f"\n  cuDF DataFrame info:")
    print(f"    shape   : {df_cd.shape}")
    print(f"    memory  : {df_cd.memory_usage(deep=True).sum() / 1e6:.1f} MB  (GPU HBM2e)")

    # ── Common operations — identical API ─────────────────────────────────
    print("\n  cuDF head():")
    print(df_cd.head(3).to_pandas())

    print("\n  cuDF describe():")
    print(df_cd["amount"].describe().to_pandas())

    # ── Filtering ─────────────────────────────────────────────────────────
    high_value = df_cd[df_cd["amount"] > 400]
    print(f"\n  Rows with amount > 400: {len(high_value):,}  ({len(high_value)/len(df_cd)*100:.1f}%)")

    # ── String columns ────────────────────────────────────────────────────
    df_cd["category"] = cudf.Series(
        np.random.choice(["electronics", "clothing", "food", "books"], size=1_000_000)
    )
    unique_cats = df_cd["category"].unique()
    print(f"\n  Unique categories: {unique_cats.to_arrow().to_pylist()}")

In [ ]:
# ── MODULE 5 · Cell 2 ── cuDF vs Pandas Benchmark ──────────────────────────
if HAS_CUDF:
    N_BENCH = 5_000_000   # 5M rows
    bench_data = {
        "user_id":  np.random.randint(0, 100_000, N_BENCH),
        "category": np.random.choice(["A","B","C","D","E","F","G","H"], N_BENCH),
        "amount":   np.random.rand(N_BENCH).astype(np.float32) * 1000,
        "cnt":      np.random.randint(1, 20, N_BENCH),
    }

    df_pd2 = pd.DataFrame(bench_data)
    df_cd2 = cudf.DataFrame(bench_data)

    # ── Benchmark helper ──────────────────────────────────────────────────
    def bench(fn, reps=5):
        for _ in range(2): fn()   # warm-up
        t0 = time.perf_counter()
        for _ in range(reps): fn()
        return (time.perf_counter() - t0) / reps * 1000

    operations = {}

    # GroupBy + aggregation
    operations["GroupBy + agg\n(user_id, 3 aggs)"] = (
        bench(lambda: df_pd2.groupby("user_id")["amount"].agg(["mean","sum","std"])),
        bench(lambda: df_cd2.groupby("user_id")["amount"].agg(["mean","sum","std"])),
    )
    # Sort
    operations["Sort\n(amount desc)"] = (
        bench(lambda: df_pd2.sort_values("amount", ascending=False)),
        bench(lambda: df_cd2.sort_values("amount", ascending=False)),
    )
    # String filter
    operations["String filter\n(category == 'A')"] = (
        bench(lambda: df_pd2[df_pd2["category"] == "A"]),
        bench(lambda: df_cd2[df_cd2["category"] == "A"]),
    )
    # Apply + aggregate
    operations["Apply rolling\n(sum, window=10)"] = (
        bench(lambda: df_pd2["amount"].rolling(10).sum(), reps=3),
        bench(lambda: df_cd2["amount"].rolling(10).sum(), reps=3),
    )
    # Merge (self join)
    right_pd = df_pd2[["user_id","cnt"]].drop_duplicates("user_id").rename(columns={"cnt":"cnt2"})
    right_cd = df_cd2[["user_id","cnt"]].drop_duplicates("user_id").rename(columns={"cnt":"cnt2"})
    operations["Merge / join\n(user_id key)"] = (
        bench(lambda: df_pd2.merge(right_pd, on="user_id"), reps=3),
        bench(lambda: df_cd2.merge(right_cd, on="user_id"), reps=3),
    )

    # Print results
    print(f"\n{'─'*68}")
    print(f"  {'Operation':<28} {'Pandas (ms)':>14} {'cuDF (ms)':>12} {'Speedup':>8}")
    print(f"{'─'*68}")
    for op, (pd_t, cd_t) in operations.items():
        op_s = op.replace("\n", " ")
        print(f"  {op_s:<28} {pd_t:>14.2f} {cd_t:>12.2f} {pd_t/cd_t:>7.1f}×")
    print(f"{'─'*68}")

    # Visualization
    ops_labels  = [k.replace("\n", "\n") for k in operations]
    pd_times    = [v[0] for v in operations.values()]
    cd_times    = [v[1] for v in operations.values()]
    speedups_5  = [p/c for p, c in zip(pd_times, cd_times)]

    x5 = np.arange(len(ops_labels))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    w = 0.35
    ax1.bar(x5 - w/2, pd_times, w, label="Pandas (CPU)", color="#e74c3c", edgecolor="white")
    ax1.bar(x5 + w/2, cd_times, w, label="cuDF (A100)",  color=NVIDIA_GREEN, edgecolor="white")
    ax1.set_xticks(x5); ax1.set_xticklabels(ops_labels, fontsize=8)
    ax1.set_ylabel("Latency (ms) — lower is better"); ax1.set_title("cuDF vs Pandas: Latency", fontweight="bold")
    ax1.set_yscale("log"); ax1.legend()

    ax2.bar(x5, speedups_5, color=NVIDIA_GREEN, edgecolor="white")
    ax2.set_xticks(x5); ax2.set_xticklabels(ops_labels, fontsize=8)
    ax2.set_ylabel("Speedup (×)"); ax2.set_title("cuDF Speedup over Pandas (A100)", fontweight="bold")
    for xi, s in zip(x5, speedups_5):
        ax2.text(xi, s + 0.5, f"{s:.0f}×", ha="center", fontsize=9, fontweight="bold")
    ax2.axhline(1, color="red", linestyle="--", linewidth=1)

    plt.tight_layout()
    plt.savefig("m5_cudf_benchmark.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("cuDF not available.")

In [ ]:
# ── MODULE 5 · Cell 3 ── cudf.pandas: Zero-Code-Change Acceleration ─────────
# cudf.pandas patches pandas itself so your existing scripts run on GPU.
# Enable it ONCE at the top of your script; no other code changes needed.

if HAS_CUDF:
    # This is how you use it in a fresh script:
    # ┌─────────────────────────────────────────────────────────┐
    # │  import cudf.pandas                                      │
    # │  cudf.pandas.install()    ← add just this line          │
    # │  import pandas as pd      ← same as always              │
    # │  ...rest of pandas code unchanged...                     │
    # └─────────────────────────────────────────────────────────┘
    #
    # Here we demonstrate by calling functions that use vanilla pandas syntax,
    # but pass cuDF DataFrames (which support the same API):

    def process_sales(df):
        """Pure pandas-style code — works with both pandas and cuDF DataFrames."""
        # GroupBy + multi-agg
        summary = df.groupby("category").agg(
            total_revenue=("amount", "sum"),
            avg_order=("amount", "mean"),
            num_orders=("amount", "count"),
        ).reset_index()
        # Derived column
        summary["revenue_pct"] = summary["total_revenue"] / summary["total_revenue"].sum() * 100
        # Sort
        summary = summary.sort_values("total_revenue", ascending=False)
        return summary

    N_DEMO = 2_000_000
    demo_data = {
        "category": np.random.choice(["Electronics","Clothing","Food","Books","HomeGarden"], N_DEMO),
        "amount":   np.random.exponential(scale=100, size=N_DEMO).astype(np.float32),
    }

    # Same function on both:
    df_pandas = pd.DataFrame(demo_data)
    df_cudf   = cudf.DataFrame(demo_data)

    t0 = time.perf_counter(); res_pd  = process_sales(df_pandas); t_pd = (time.perf_counter()-t0)*1000
    t0 = time.perf_counter(); res_cd  = process_sales(df_cudf);   t_cd = (time.perf_counter()-t0)*1000

    print(f"  Same function, same syntax:")
    print(f"    Pandas (CPU)  : {t_pd:.1f} ms")
    print(f"    cuDF   (A100) : {t_cd:.1f} ms  ({t_pd/t_cd:.1f}× speedup)")
    print("\n  GPU result:")
    print(res_cd.to_pandas().to_string(index=False))
    print("\n  💡 With cudf.pandas.install(), ALL your pandas code accelerates automatically.")
    print("  Zero code changes needed for most workloads!")
else:
    print("cuDF not available.")

# ═══════════════════════════════════════════════════════════════
# MODULE 6 — RAPIDS cuML: GPU Machine Learning
# ═══════════════════════════════════════════════════════════════

## What is cuML?

**cuML** is RAPIDS' GPU-accelerated machine learning library. It mirrors **scikit-learn's API** — same `.fit()`, `.predict()`, `.transform()` calls, but everything runs on the GPU.

```python
# scikit-learn (CPU)
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# cuML (GPU) — identical API!
from cuml.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train_gpu, y_train_gpu)
```

## Supported Algorithms (selection)

| Category | cuML Algorithm | sklearn Equivalent |
|----------|---------------|-------------------|
| Regression | `cuml.LinearRegression` | `sklearn.linear_model.LinearRegression` |
| Classification | `cuml.RandomForestClassifier` | `sklearn.ensemble.RandomForestClassifier` |
| Clustering | `cuml.KMeans`, `cuml.DBSCAN` | `sklearn.cluster.KMeans`, `DBSCAN` |
| Nearest Neighbors | `cuml.NearestNeighbors` | `sklearn.neighbors.NearestNeighbors` |
| Dimensionality | `cuml.UMAP`, `cuml.PCA`, `cuml.TSNE` | `umap`, `sklearn.decomposition.PCA` |
| Preprocessing | `cuml.StandardScaler` | `sklearn.preprocessing.StandardScaler` |

## Speedup Source

cuML kernels are written in CUDA C and exploit GPU parallelism at every stage:
- KMeans: all distances computed in parallel across points and centroids
- RandomForest: all trees trained in parallel (different bootstraps)
- KNN: brute-force K nearest neighbors using cuBLAS matmul tricks

In [ ]:
# ── MODULE 6 · Cell 1 ── cuML Setup & API Demo ─────────────────────────────
try:
    import cuml
    from cuml.linear_model import LinearRegression as cuLinearRegression
    from cuml.neighbors import KNeighborsClassifier as cuKNN
    from cuml.cluster import KMeans as cuKMeans, DBSCAN as cuDBSCAN
    from cuml.ensemble import RandomForestClassifier as cuRFC
    from cuml.preprocessing import StandardScaler as cuScaler
    from cuml.model_selection import train_test_split as cu_tts
    HAS_CUML = True
    print(f"cuML version: {cuml.__version__}")
except ImportError:
    HAS_CUML = False
    print("⚠️  cuML not installed. Install via: pip install cuml-cu12 --extra-index-url https://pypi.nvidia.com")

from sklearn.linear_model import LinearRegression as skLinReg
from sklearn.neighbors import KNeighborsClassifier as skKNN
from sklearn.cluster import KMeans as skKMeans
from sklearn.ensemble import RandomForestClassifier as skRFC
from sklearn.preprocessing import StandardScaler as skScaler
from sklearn.model_selection import train_test_split as sk_tts
from sklearn.datasets import make_classification, make_regression, make_blobs
from sklearn.metrics import accuracy_score

print(f"scikit-learn version: {__import__('sklearn').__version__}")

In [ ]:
# ── MODULE 6 · Cell 2 ── cuML vs sklearn: Algorithm Benchmarks ─────────────
def time_fn(fn, reps=3):
    for _ in range(1): fn()  # warm-up
    t0 = time.perf_counter()
    for _ in range(reps): result = fn()
    return (time.perf_counter() - t0) / reps * 1000, result

ml_results = {}

# ── Dataset sizes to test ─────────────────────────────────────────────────
N_CLASS = 100_000
N_FEAT  = 64

print("Generating datasets...")
X_cls, y_cls = make_classification(n_samples=N_CLASS, n_features=N_FEAT,
                                   n_informative=32, random_state=42)
X_cls = X_cls.astype(np.float32);  y_cls = y_cls.astype(np.int32)

X_reg, y_reg = make_regression(n_samples=N_CLASS, n_features=N_FEAT,
                                n_informative=32, random_state=42)
X_reg = X_reg.astype(np.float32);  y_reg = y_reg.astype(np.float32)

X_kmeans, _ = make_blobs(n_samples=N_CLASS, n_features=N_FEAT,
                          centers=20, random_state=42)
X_kmeans = X_kmeans.astype(np.float32)

# CPU train/test split
X_tr_cpu, X_te_cpu, y_tr_cpu, y_te_cpu = sk_tts(X_cls, y_cls, test_size=0.2, random_state=42)

print("Running benchmarks...")

# ── 1. KMeans ─────────────────────────────────────────────────────────────
t_sk, _ = time_fn(lambda: skKMeans(n_clusters=20, max_iter=100, n_init=1, random_state=42).fit(X_kmeans))
if HAS_CUML:
    X_km_gpu = cudf.DataFrame(X_kmeans)
    t_cu, _ = time_fn(lambda: cuKMeans(n_clusters=20, max_iter=100).fit(X_km_gpu))
    ml_results["KMeans\n(20 clusters)"] = (t_sk, t_cu)
else:
    ml_results["KMeans\n(20 clusters)"] = (t_sk, None)

# ── 2. Linear Regression ──────────────────────────────────────────────────
t_sk, _ = time_fn(lambda: skLinReg().fit(X_reg, y_reg))
if HAS_CUML:
    X_r_gpu = cudf.DataFrame(X_reg);  y_r_gpu = cudf.Series(y_reg)
    t_cu, _ = time_fn(lambda: cuLinearRegression().fit(X_r_gpu, y_r_gpu))
    ml_results["Linear Regression"] = (t_sk, t_cu)
else:
    ml_results["Linear Regression"] = (t_sk, None)

# ── 3. KNN Classification ─────────────────────────────────────────────────
t_sk, _ = time_fn(lambda: skKNN(n_neighbors=10).fit(X_tr_cpu, y_tr_cpu), reps=1)
if HAS_CUML:
    X_tr_g = cudf.DataFrame(X_tr_cpu);  y_tr_g = cudf.Series(y_tr_cpu)
    t_cu, _ = time_fn(lambda: cuKNN(n_neighbors=10).fit(X_tr_g, y_tr_g), reps=1)
    ml_results["KNN\n(k=10, 80K train)"] = (t_sk, t_cu)
else:
    ml_results["KNN\n(k=10, 80K train)"] = (t_sk, None)

# ── 4. Random Forest ──────────────────────────────────────────────────────
t_sk, _ = time_fn(lambda: skRFC(n_estimators=100, max_depth=8, n_jobs=-1, random_state=42).fit(X_tr_cpu, y_tr_cpu), reps=1)
if HAS_CUML:
    X_tr_g2 = cudf.DataFrame(X_tr_cpu);  y_tr_g2 = cudf.Series(y_tr_cpu.astype(np.float32))
    t_cu, _ = time_fn(lambda: cuRFC(n_estimators=100, max_depth=8, random_state=42).fit(X_tr_g2, y_tr_g2), reps=1)
    ml_results["Random Forest\n(100 trees)"] = (t_sk, t_cu)
else:
    ml_results["Random Forest\n(100 trees)"] = (t_sk, None)

# ── Print results ─────────────────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"  {'Algorithm':<28} {'sklearn (ms)':>14} {'cuML (ms)':>12} {'Speedup':>8}")
print(f"{'─'*65}")
for alg, (sk_t, cu_t) in ml_results.items():
    alg_s = alg.replace("\n", " ")
    if cu_t is not None:
        print(f"  {alg_s:<28} {sk_t:>14.1f} {cu_t:>12.1f} {sk_t/cu_t:>7.1f}×")
    else:
        print(f"  {alg_s:<28} {sk_t:>14.1f} {'N/A':>12}  {'N/A':>7}")
print(f"{'─'*65}")
print(f"\n  Dataset: N={N_CLASS:,}  features={N_FEAT}")

# Speedup heatmap (if cuML available)
if HAS_CUML:
    speedups_ml  = {k.replace("\n"," "): v[0]/v[1] for k, v in ml_results.items() if v[1] is not None}
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(list(speedups_ml.keys()), list(speedups_ml.values()),
                   color=NVIDIA_GREEN, edgecolor="white")
    ax.set_xlabel("Speedup (×) over scikit-learn — higher is better")
    ax.set_title(f"cuML vs scikit-learn Speedup (N={N_CLASS:,}, {N_FEAT} features, A100 GPU)",
                 fontweight="bold")
    ax.axvline(1, color="red", linestyle="--", linewidth=1, label="Breakeven")
    for bar, v in zip(bars, speedups_ml.values()):
        ax.text(v + 0.5, bar.get_y() + bar.get_height()/2,
                f"{v:.1f}×", va="center", fontsize=10, fontweight="bold")
    ax.legend(); plt.tight_layout()
    plt.savefig("m6_cuml_speedup.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── MODULE 6 · Cell 3 ── GPU Dimensionality Reduction: cuML UMAP & PCA ──────
if HAS_CUML:
    from cuml.decomposition import PCA as cuPCA
    from cuml.manifold import UMAP as cuUMAP
    from sklearn.decomposition import PCA as skPCA

    N_VIZ = 20_000
    X_viz, y_viz = make_classification(n_samples=N_VIZ, n_features=50,
                                        n_informative=20, n_classes=5, random_state=42)
    X_viz = X_viz.astype(np.float32)

    X_viz_gpu = cudf.DataFrame(X_viz)

    # ── PCA ─────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    pca_sk = skPCA(n_components=2).fit_transform(X_viz)
    t_pca_sk = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    pca_cu = cuPCA(n_components=2).fit_transform(X_viz_gpu)
    torch.cuda.synchronize()
    t_pca_cu = (time.perf_counter() - t0) * 1000
    pca_cu_np = pca_cu.to_numpy()

    print(f"  PCA (N={N_VIZ:,}, 50→2D): sklearn={t_pca_sk:.1f}ms | cuML={t_pca_cu:.1f}ms | {t_pca_sk/t_pca_cu:.1f}× speedup")

    # ── UMAP ────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    umap_embedding = cuUMAP(n_neighbors=15, n_components=2, random_state=42).fit_transform(X_viz_gpu)
    torch.cuda.synchronize()
    t_umap = (time.perf_counter() - t0) * 1000
    umap_np = umap_embedding.to_numpy()

    print(f"  UMAP (N={N_VIZ:,}, 50→2D): cuML GPU = {t_umap:.0f}ms")

    # Visualize both
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for ax, embed, title in [(ax1, pca_cu_np, f"cuML PCA ({t_pca_cu:.0f}ms)"),
                              (ax2, umap_np,   f"cuML UMAP ({t_umap:.0f}ms)")]:
        scatter = ax.scatter(embed[:, 0], embed[:, 1], c=y_viz,
                             cmap="tab10", s=3, alpha=0.6)
        plt.colorbar(scatter, ax=ax, label="Class")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Component 1"); ax.set_ylabel("Component 2")

    fig.suptitle(f"GPU Dimensionality Reduction with cuML (N={N_VIZ:,})", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("m6_umap_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n💡 UMAP often produces better cluster separation than PCA (nonlinear).")
    print("   cuML UMAP is typically 10–50× faster than the CPU umap-learn package.")
else:
    print("cuML not available.")

# ═══════════════════════════════════════════════════════════════
# MODULE 7 — RAPIDS cuGraph: GPU Graph Analytics
# ═══════════════════════════════════════════════════════════════

## What is cuGraph?

**cuGraph** is RAPIDS' GPU-accelerated graph analytics library. Graph algorithms are notoriously difficult to parallelize due to irregular access patterns, but cuGraph implements highly optimized CUDA kernels for the most common graph algorithms.

## Key Algorithms Covered

| Algorithm | Use Case | Complexity |
|-----------|----------|------------|
| **PageRank** | Ranking nodes by importance (like Google) | O(V + E) per iter |
| **BFS** | Shortest path in unweighted graph | O(V + E) |
| **SSSP** | Shortest path in weighted graph (Dijkstra) | O((V+E) log V) |
| **Louvain** | Community detection (clustering) | O(V log V) |
| **Betweenness Centrality** | Find bridge/hub nodes | O(VE) |

## Graph Representation

cuGraph uses a **CSR (Compressed Sparse Row)** representation internally, stored in GPU memory. You build the graph from a cuDF edge list:

```python
import cugraph
G = cugraph.Graph()
G.from_cudf_edgelist(edge_df, source="src", destination="dst", edge_attr="weight")
```

In [ ]:
# ── MODULE 7 · Cell 1 ── cuGraph Setup & Graph Construction ────────────────
try:
    import cugraph
    HAS_CUGRAPH = True
    print(f"cuGraph version: {cugraph.__version__}")
except ImportError:
    HAS_CUGRAPH = False
    print("⚠️  cuGraph not installed. Install via: pip install cugraph-cu12 --extra-index-url https://pypi.nvidia.com")

import networkx as nx
print(f"NetworkX version: {nx.__version__}")

# Helper: generate a random graph edge list
def make_random_edges(n_nodes, n_edges, seed=42):
    rng = np.random.default_rng(seed)
    src = rng.integers(0, n_nodes, n_edges).astype(np.int32)
    dst = rng.integers(0, n_nodes, n_edges).astype(np.int32)
    # Remove self-loops
    mask = src != dst
    src, dst = src[mask], dst[mask]
    weights = rng.random(len(src)).astype(np.float32)
    return src, dst, weights

# Demo graph: 10K nodes, 100K edges
N_NODES  = 10_000
N_EDGES  = 100_000
src, dst, weights = make_random_edges(N_NODES, N_EDGES)

print(f"\n  Graph: {N_NODES:,} nodes, {len(src):,} edges")

if HAS_CUGRAPH:
    # Build cuDF edge list
    edge_df = cudf.DataFrame({"src": src, "dst": dst, "weight": weights})
    G_cu = cugraph.Graph()
    G_cu.from_cudf_edgelist(edge_df, source="src", destination="dst",
                             edge_attr="weight", renumber=True)
    print(f"  cuGraph: {G_cu.number_of_vertices()} vertices, {G_cu.number_of_edges()} edges")

# NetworkX reference graph
G_nx = nx.DiGraph()
for s, d, w in zip(src[:20_000], dst[:20_000], weights[:20_000]):  # subset for NX
    G_nx.add_edge(int(s), int(d), weight=float(w))
print(f"  NetworkX: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges (20K subset)")

In [ ]:
# ── MODULE 7 · Cell 2 ── cuGraph Algorithm Benchmarks ──────────────────────
graph_results = {}

if HAS_CUGRAPH:
    # ── PageRank ──────────────────────────────────────────────────────────
    # GPU
    t0 = time.perf_counter()
    pr_cu = cugraph.pagerank(G_cu, alpha=0.85, max_iter=100)
    torch.cuda.synchronize()
    t_pr_cu = (time.perf_counter() - t0) * 1000

    # CPU (NetworkX — on subset)
    t0 = time.perf_counter()
    pr_nx = nx.pagerank(G_nx, alpha=0.85, max_iter=100)
    t_pr_nx = (time.perf_counter() - t0) * 1000

    print(f"  PageRank  GPU ({N_NODES:,} nodes, {N_EDGES:,} edges) : {t_pr_cu:.1f} ms")
    print(f"  PageRank  CPU (20K nodes, 20K edges)         : {t_pr_nx:.1f} ms")
    graph_results["PageRank"] = (t_pr_nx, t_pr_cu, N_NODES)

    # Top-5 nodes by PageRank
    top5 = pr_cu.sort_values("pagerank", ascending=False).head(5)
    print(f"\n  Top-5 nodes by PageRank:")
    print(top5.to_pandas().to_string(index=False))

    # ── BFS ───────────────────────────────────────────────────────────────
    G_cu_undirected = cugraph.Graph(directed=False)
    G_cu_undirected.from_cudf_edgelist(edge_df, source="src", destination="dst", renumber=True)

    t0 = time.perf_counter()
    bfs_cu = cugraph.bfs(G_cu_undirected, start_vertices=0)
    torch.cuda.synchronize()
    t_bfs_cu = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    bfs_nx = nx.bfs_tree(G_nx.to_undirected(), source=0)
    t_bfs_nx = (time.perf_counter() - t0) * 1000

    print(f"\n  BFS GPU: {t_bfs_cu:.1f} ms  |  NX CPU: {t_bfs_nx:.1f} ms")
    graph_results["BFS"] = (t_bfs_nx, t_bfs_cu, N_NODES)

    # ── Louvain Community Detection ───────────────────────────────────────
    t0 = time.perf_counter()
    parts, modularity = cugraph.louvain(G_cu_undirected)
    torch.cuda.synchronize()
    t_lou_cu = (time.perf_counter() - t0) * 1000

    n_communities = len(parts["partition"].unique())
    print(f"\n  Louvain GPU: {t_lou_cu:.1f} ms  |  Modularity: {modularity:.4f}  |  Communities: {n_communities}")
    graph_results["Louvain"] = (None, t_lou_cu, N_NODES)

    # ── Summary chart ─────────────────────────────────────────────────────
    algs    = list(graph_results.keys())
    gpu_ts  = [v[1] for v in graph_results.values()]
    cpu_ts  = [v[0] if v[0] is not None else None for v in graph_results.values()]

    fig, ax = plt.subplots(figsize=(9, 4))
    x_g = np.arange(len(algs))
    ax.bar(x_g, gpu_ts, color=NVIDIA_GREEN, edgecolor="white", label="cuGraph (GPU)")
    # CPU only where comparable
    cpu_valid = [(i, t) for i, t in enumerate(cpu_ts) if t is not None]
    if cpu_valid:
        xi, ti = zip(*cpu_valid)
        ax.bar(np.array(xi) - 0.0, ti, width=0.35, color="#e74c3c",
               edgecolor="white", label="NetworkX (CPU)", alpha=0.6)
    ax.set_xticks(x_g); ax.set_xticklabels(algs)
    ax.set_ylabel("Latency (ms)"); ax.legend()
    ax.set_title(f"cuGraph Algorithm Latency (A100, {N_NODES:,} nodes)", fontweight="bold")
    plt.tight_layout()
    plt.savefig("m7_cugraph.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("cuGraph not available — running NetworkX demo only.")
    t0 = time.perf_counter()
    pr_nx = nx.pagerank(G_nx, alpha=0.85, max_iter=100)
    t_pr_nx = (time.perf_counter() - t0) * 1000
    print(f"  NetworkX PageRank (20K nodes): {t_pr_nx:.1f} ms")

# ═══════════════════════════════════════════════════════════════
# MODULE 8 — Dask-cuDF: Multi-GPU & Out-of-Core DataFrames
# ═══════════════════════════════════════════════════════════════

## What is Dask-cuDF?

**Dask** is a parallel computing library that splits large datasets into chunks and processes them lazily. **Dask-cuDF** replaces each Dask partition (normally a Pandas DataFrame) with a **cuDF GPU DataFrame**.

This gives you:
- **Multi-GPU scaling**: each GPU handles a subset of partitions in parallel
- **Out-of-core processing**: datasets that don't fit in one GPU's HBM2e
- **Lazy evaluation**: build a computation graph, execute only when needed with `.compute()`

```
Dask-cuDF workflow:
┌────────────────────────────────────────────────────────┐
│  Large Dataset (e.g. 500 GB Parquet on disk/S3)        │
│                                                        │
│  dask_cudf.read_parquet(path)  ← lazy, no data loaded │
│         ↓                                              │
│  Partition 0 (cuDF)  Partition 1 (cuDF)  ...           │
│  [GPU 0]             [GPU 1]             [GPU n]        │
│         ↓                                              │
│  .groupby().agg()  ← each GPU processes its partition  │
│         ↓                                              │
│  .compute()  ← trigger execution, collect results      │
└────────────────────────────────────────────────────────┘
```

## Key API

```python
import dask_cudf
ddf = dask_cudf.read_parquet("s3://bucket/data/*.parquet")  # lazy
result = ddf.groupby("user_id")["amount"].sum().compute()   # execute
```

> 💡 Even on a **single A100**, Dask-cuDF is useful for datasets larger than GPU memory — it pages partitions in and out automatically.

In [ ]:
# ── MODULE 8 · Cell 1 ── Dask-cuDF Setup & Partitioned DataFrames ──────────
try:
    import dask_cudf
    import dask
    HAS_DASK_CUDF = True
    print(f"dask         version: {dask.__version__}")
    print(f"dask_cudf    version: {dask_cudf.__version__}")
except ImportError:
    HAS_DASK_CUDF = False
    print("⚠️  dask-cudf not installed. Install via: pip install dask-cudf-cu12 --extra-index-url https://pypi.nvidia.com")

if HAS_DASK_CUDF and HAS_CUDF:
    # Generate a larger-than-comfortable single-GPU dataset by splitting into partitions
    N_DASK = 10_000_000
    rng = np.random.default_rng(42)
    data_dask = {
        "user_id":  rng.integers(0, 100_000, N_DASK).astype(np.int32),
        "product":  rng.integers(0, 500,     N_DASK).astype(np.int32),
        "amount":   rng.exponential(scale=100, size=N_DASK).astype(np.float32),
        "qty":      rng.integers(1, 10,      N_DASK).astype(np.int32),
    }
    df_full_cu = cudf.DataFrame(data_dask)

    # ── Create Dask-cuDF DataFrame with N partitions ───────────────────────
    N_PARTS = 8
    ddf = dask_cudf.from_cudf(df_full_cu, npartitions=N_PARTS)

    print(f"\n  Dask-cuDF DataFrame:")
    print(f"    Total rows       : {len(df_full_cu):,}")
    print(f"    Partitions       : {ddf.npartitions}")
    print(f"    Rows/partition   : ~{len(df_full_cu) // N_PARTS:,}")
    print(f"    Memory / part    : ~{df_full_cu.memory_usage(deep=True).sum() / N_PARTS / 1e6:.0f} MB")
    print(f"\n  Schema:")
    print(ddf.dtypes.to_string())

    # ── Lazy computation graph ─────────────────────────────────────────────
    lazy_result = ddf.groupby("user_id")["amount"].agg(["mean", "sum", "count"])
    print(f"\n  Lazy computation graph built (not yet executed):")
    print(f"  Type: {type(lazy_result)}")

    # ── Execute the graph ─────────────────────────────────────────────────
    t0 = time.perf_counter()
    result = lazy_result.compute()
    t_dask = (time.perf_counter() - t0) * 1000

    # Compare to raw cuDF
    t0 = time.perf_counter()
    result_cudf = df_full_cu.groupby("user_id")["amount"].agg(["mean", "sum", "count"])
    t_cudf = (time.perf_counter() - t0) * 1000

    print(f"\n  GroupBy result (top 5 rows):")
    print(result.sort_values("sum", ascending=False).head(5).to_pandas().to_string())
    print(f"\n  Timing ({N_DASK:,} rows, {N_PARTS} partitions):")
    print(f"    Dask-cuDF .compute() : {t_dask:.1f} ms")
    print(f"    Single cuDF          : {t_cudf:.1f} ms")
    print(f"\n  💡 Dask overhead is small for large datasets.")
    print(f"     Benefit: can process datasets larger than a single GPU's HBM (paging across partitions).")
    print(f"     With multiple GPUs: partitions run in parallel across GPUs, scaling linearly.")
else:
    print("Dask-cuDF not available.")

In [ ]:
# ── MODULE 8 · Cell 2 ── Multi-GPU Architecture Diagram ────────────────────
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(0, 13); ax.set_ylim(0, 8); ax.axis("off")
ax.set_title("Dask-cuDF Multi-GPU Data Parallelism Architecture", fontsize=14, fontweight="bold")

# Storage
draw_box(ax, 0.3, 0.2, 12.4, 0.9, "Storage: Parquet / CSV on Disk / S3 / HDFS  (100s GB – TB scale)",
         "#7f8c8d", fontsize=9)

# Arrow up
ax.annotate("", xy=(6.5, 2.0), xytext=(6.5, 1.2),
            arrowprops=dict(arrowstyle="->", color="#333", lw=2))
ax.text(7.0, 1.6, "dask_cudf.read_parquet()\nReads lazily, splits into partitions", fontsize=8, color="#333")

# Dask scheduler
draw_box(ax, 4.5, 2.0, 4.0, 0.85, "Dask Scheduler (CPU)\nBuilds & orchestrates compute graph", "#8e44ad", fontsize=8)

# GPU workers
gpu_colors = [NVIDIA_GREEN, "#2980b9", "#e67e22", "#c0392b"]
gpu_labels = ["GPU 0 (A100)\nPartitions 0,1", "GPU 1 (A100)\nPartitions 2,3",
              "GPU 2 (A100)\nPartitions 4,5", "GPU 3 (A100)\nPartitions 6,7"]
for i in range(4):
    bx = 0.3 + i * 3.15
    draw_box(ax, bx, 3.3, 2.85, 1.6, gpu_labels[i], gpu_colors[i], fontsize=8)
    ax.annotate("", xy=(bx+1.42, 3.3), xytext=(bx+1.42, 2.85),
                arrowprops=dict(arrowstyle="->", color="#555", lw=1.5))
    # Partition boxes inside each GPU
    for p in range(2):
        px = bx + 0.1 + p * 1.4
        rect = plt.Rectangle((px, 3.45), 1.25, 0.5, fc="white", ec=gpu_colors[i], lw=1, alpha=0.8)
        ax.add_patch(rect)
        ax.text(px+0.625, 3.7, f"cuDF partition\n{i*2+p}", ha="center", fontsize=6.5, color=gpu_colors[i])

# NVLink arch note
ax.text(6.5, 5.35,
        "NVLink 3.0: 600 GB/s bidirectional between GPUs\n"
        "PCIe 4.0: ~32 GB/s to CPU/storage",
        ha="center", fontsize=9, style="italic", color="#444",
        bbox=dict(boxstyle="round,pad=0.4", fc="#fef9e7", ec="#f39c12"))

# Aggregation arrow
ax.annotate("", xy=(6.5, 5.9), xytext=(6.5, 5.7),
            arrowprops=dict(arrowstyle="->", color="#333", lw=2))

draw_box(ax, 3.5, 5.9, 6.0, 0.85, ".compute()  → Collect & reduce across GPUs → Result", "#2c3e50", fontsize=9)

ax.text(6.5, 7.1,
        "Scales to 8× A100s (DGX A100) with NVLink | Handles TB-scale datasets",
        ha="center", fontsize=10, fontweight="bold", color="#2c3e50")

plt.tight_layout()
plt.savefig("m8_dask_cudf_arch.png", dpi=150, bbox_inches="tight")
plt.show()

# ═══════════════════════════════════════════════════════════════
# MODULE 9 — PyTorch GPU Deep Dive
# ═══════════════════════════════════════════════════════════════

## What We Cover

PyTorch hides most GPU complexity behind a clean Python API, but understanding the internals unlocks significant performance gains:

1. **CUDA Streams** — run compute and data transfer simultaneously
2. **Non-blocking transfers** — don't wait for H2D before launching kernels
3. **Custom autograd functions** — write GPU-backed ops with custom gradients
4. **`torch.compile()`** — trace your model, fuse kernels, auto-tune

## CUDA Streams

By default, all PyTorch GPU operations run on the **default stream** — sequentially.  
With multiple streams, operations on different streams can **overlap**:

```
Default stream (sequential):
  [transfer A] → [kernel 1] → [transfer B] → [kernel 2]
  
Multi-stream (parallel transfers + compute):
  Stream 1: [transfer A] → [kernel 1]
  Stream 2:   [transfer B] → [kernel 2]         ← overlap!
```

This is critical when you have a DataLoader pre-fetching batches while the GPU is training on the current batch.

## torch.compile() (PyTorch 2.0+)

`torch.compile()` uses **TorchDynamo** (Python bytecode tracing) + **TorchInductor** (kernel code generation) to:
- Fuse consecutive elementwise ops into a single kernel
- Generate optimized Triton or CUDA C kernels
- Eliminate Python overhead in tight loops

```python
model = torch.compile(model)   # that's it!
# First call: ~seconds to compile (JIT)
# Subsequent calls: compiled + fused kernels
```

In [ ]:
# ── MODULE 9 · Cell 1 ── CUDA Streams: Overlapping Transfer & Compute ───────
if torch.cuda.is_available():
    N_BATCHES  = 8
    BATCH_SIZE = 512
    INPUT_DIM  = 1024
    HIDDEN_DIM = 2048

    model = nn.Sequential(
        nn.Linear(INPUT_DIM, HIDDEN_DIM), nn.ReLU(),
        nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
        nn.Linear(HIDDEN_DIM, 256)
    ).cuda().eval()

    # Simulate batches waiting on CPU (pinned for async transfer)
    cpu_batches = [torch.randn(BATCH_SIZE, INPUT_DIM).pin_memory() for _ in range(N_BATCHES)]

    # ── Sequential baseline (default stream) ─────────────────────────────
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        results_seq = []
        for batch in cpu_batches:
            x = batch.cuda()                    # blocking H2D
            torch.cuda.synchronize()
            out = model(x)
            torch.cuda.synchronize()
            results_seq.append(out)
    t_seq = (time.perf_counter() - t0) * 1000

    # ── Multi-stream: transfer + compute overlap ──────────────────────────
    streams = [torch.cuda.Stream() for _ in range(2)]
    gpu_bufs = [torch.empty(BATCH_SIZE, INPUT_DIM, device="cuda") for _ in range(2)]
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        results_stream = []
        for i, batch in enumerate(cpu_batches):
            stream = streams[i % 2]
            buf    = gpu_bufs[i % 2]
            with torch.cuda.stream(stream):
                buf.copy_(batch, non_blocking=True)   # async H2D on this stream
                out = model(buf)                       # compute begins as soon as copy done
                results_stream.append(out.clone())
        # Wait for all streams
        for s in streams:
            torch.cuda.current_stream().wait_stream(s)
    torch.cuda.synchronize()
    t_stream = (time.perf_counter() - t0) * 1000

    print(f"\n  CUDA Streams Benchmark  ({N_BATCHES} batches × {BATCH_SIZE} samples)")
    print(f"  Sequential (default stream)  : {t_seq:.2f} ms")
    print(f"  Multi-stream (2 streams)     : {t_stream:.2f} ms   ({t_seq/t_stream:.2f}× speedup)")
    print("\n  💡 Speedup grows with more batches and larger transfer sizes.")
    print("     PyTorch DataLoader uses pin_memory + non_blocking=True by default.")

    del model, cpu_batches, gpu_bufs
    torch.cuda.empty_cache()
else:
    print("No GPU.")

In [ ]:
# ── MODULE 9 · Cell 2 ── torch.compile() Optimization (PyTorch 2.0+) ────────
# torch.compile uses TorchDynamo + TorchInductor to fuse ops and generate optimized kernels.

if torch.cuda.is_available():
    class SmallTransformerBlock(nn.Module):
        """A single Transformer block representative of LLM building blocks."""
        def __init__(self, d_model=512, n_heads=8, ffn_dim=2048):
            super().__init__()
            self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.ffn = nn.Sequential(
                nn.Linear(d_model, ffn_dim), nn.GELU(),
                nn.Linear(ffn_dim, d_model)
            )
        def forward(self, x):
            attn_out, _ = self.attn(x, x, x)
            x = self.norm1(x + attn_out)
            x = self.norm2(x + self.ffn(x))
            return x

    class SmallTransformer(nn.Module):
        def __init__(self, n_layers=6, d_model=512):
            super().__init__()
            self.layers = nn.ModuleList([SmallTransformerBlock(d_model) for _ in range(n_layers)])
        def forward(self, x):
            for layer in self.layers:
                x = layer(x)
            return x

    SEQ_LEN = 128; BATCH  = 32; D_MODEL = 512
    x_input = torch.randn(BATCH, SEQ_LEN, D_MODEL, device="cuda")

    model_eager   = SmallTransformer().cuda().eval()
    model_compiled = torch.compile(SmallTransformer().cuda().eval(), mode="reduce-overhead")

    def bench_model(model, x, n_warmup=5, n_reps=50):
        with torch.no_grad():
            for _ in range(n_warmup):
                _ = model(x)
            torch.cuda.synchronize()
            start = torch.cuda.Event(enable_timing=True)
            end   = torch.cuda.Event(enable_timing=True)
            start.record()
            for _ in range(n_reps):
                _ = model(x)
            end.record()
            torch.cuda.synchronize()
        return start.elapsed_time(end) / n_reps

    print("Benchmarking eager model...")
    t_eager = bench_model(model_eager, x_input)

    print("Benchmarking compiled model (first call triggers compilation, may take ~30s)...")
    t_compiled = bench_model(model_compiled, x_input)

    speedup = t_eager / t_compiled
    print(f"\n  Transformer ({6} layers, d_model={D_MODEL}, seq_len={SEQ_LEN}, batch={BATCH})")
    print(f"  Eager                : {t_eager:.3f} ms/iter")
    print(f"  torch.compile()      : {t_compiled:.3f} ms/iter  ({speedup:.2f}× speedup)")

    # Compile modes
    print("\n  torch.compile() modes:")
    print("    'default'         : Balance between compile time and runtime performance")
    print("    'reduce-overhead' : Minimize kernel launch overhead (good for small ops)")
    print("    'max-autotune'    : Maximize throughput (longest compile, best runtime)")
    print("\n  💡 On A100, torch.compile typically gives 10–40% speedup for transformer models.")
    print("     It is especially effective with BF16 AMP (Module 10).")

    del model_eager, model_compiled
    torch.cuda.empty_cache()
else:
    print("No GPU.")

# ═══════════════════════════════════════════════════════════════
# MODULE 10 — A100 Tensor Cores & Mixed Precision Training
# ═══════════════════════════════════════════════════════════════

## The Tensor Core Advantage on A100

NVIDIA A100 has **432 3rd-generation Tensor Cores**. A Tensor Core performs a $D = A \times B + C$ matrix operation (a "warp-level matrix multiply") in a **single clock cycle** instead of many.

| Precision | Theory TFLOPS | Notes |
|-----------|--------------|-------|
| FP32 (CUDA cores) | ~19.5 | Baseline |
| TF32 (Tensor Core) | ~156 | **A100 DEFAULT** for `torch.matmul`/`nn.Linear` |
| BF16 (Tensor Core) | ~312 | Preferred for LLM training |
| FP16 (Tensor Core) | ~312 | Same as BF16 but narrower exponent |
| INT8 (Tensor Core) | ~624 | Inference only |

## TF32 vs FP32

TF32 uses **19-bit mantissa → 10-bit** (same as FP16) internally but stores in FP32 format. The A100 automatically uses TF32 Tensor Cores for `torch.matmul` unless you disable it:

```python
torch.backends.cuda.matmul.allow_tf32 = True   # default on A100 (Ampere+)
torch.backends.cudnn.allow_tf32       = True   # default on A100
```

## BF16 vs FP16: Why A100 Prefers BF16

| Property | FP32 | FP16 | BF16 |
|----------|------|------|------|
| Bits | 32 | 16 | 16 |
| Exponent bits | 8 | 5 | **8** |
| Mantissa bits | 23 | 10 | 7 |
| Dynamic range | FP32 | **Narrow** | **FP32** |
| Risk of overflow | None | **Common** (needs GradScaler) | Rare |

**BF16 has the same dynamic range as FP32** but with lower precision — much safer for training large models. A100 fully supports BF16 Tensor Cores natively.

## Automatic Mixed Precision (AMP)

```python
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()        # only needed for FP16 (not BF16 on A100)

for batch in dataloader:
    optimizer.zero_grad()
    with autocast(dtype=torch.bfloat16):    # ← ops inside run in BF16
        output = model(batch)
        loss   = criterion(output, labels)
    # scaler.scale(loss).backward()         # FP16 only
    loss.backward()                         # BF16: no scaling needed
    optimizer.step()
```

In [ ]:
# ── MODULE 10 · Cell 1 ── Precision Benchmarks: FP32 / TF32 / FP16 / BF16 ──
if torch.cuda.is_available():
    # Enable/disable TF32
    def set_precision(allow_tf32=True):
        torch.backends.cuda.matmul.allow_tf32 = allow_tf32
        torch.backends.cudnn.allow_tf32       = allow_tf32

    SIZE = 4096
    REPS = 100

    def matmul_time_ms(dtype, n_warmup=10):
        A = torch.randn(SIZE, SIZE, device="cuda", dtype=dtype)
        B = torch.randn(SIZE, SIZE, device="cuda", dtype=dtype)
        for _ in range(n_warmup): torch.mm(A, B)
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(REPS): torch.mm(A, B)
        e.record(); torch.cuda.synchronize()
        ms = s.elapsed_time(e) / REPS
        flops = 2 * SIZE**3
        tflops = flops / (ms/1000) / 1e12
        return ms, tflops

    precision_results = {}

    set_precision(allow_tf32=False)
    ms, tflops = matmul_time_ms(torch.float32)
    precision_results["FP32\n(CUDA cores)"] = (ms, tflops)

    set_precision(allow_tf32=True)    # TF32 Tensor Cores on A100
    ms, tflops = matmul_time_ms(torch.float32)
    precision_results["TF32\n(Tensor Core, default)"] = (ms, tflops)

    ms, tflops = matmul_time_ms(torch.float16)
    precision_results["FP16\n(Tensor Core)"] = (ms, tflops)

    ms, tflops = matmul_time_ms(torch.bfloat16)
    precision_results["BF16\n(Tensor Core)"] = (ms, tflops)

    print(f"\n{'─'*65}")
    print(f"  Matrix Multiply {SIZE}×{SIZE} | A100 GPU")
    print(f"{'─'*65}")
    print(f"  {'Precision':<30} {'Latency (ms)':>14} {'TFLOPS':>10} {'vs FP32':>8}")
    fp32_ms = precision_results["FP32\n(CUDA cores)"][0]
    for lbl, (ms, tflops) in precision_results.items():
        lbl_s = lbl.replace("\n", " ")
        print(f"  {lbl_s:<30} {ms:>14.3f} {tflops:>10.1f} {fp32_ms/ms:>7.1f}×")
    print(f"{'─'*65}")
    print("\n  A100 Peak:  TF32 ~156 TFLOPS | BF16/FP16 TC ~312 TFLOPS")
    print("  → Precision matters enormously for throughput!")

    # Visualization
    lbls    = [k.replace("\n", "\n") for k in precision_results]
    tflops_vals = [v[1] for v in precision_results.values()]
    colors_prec = ["#e74c3c", "#f39c12", "#3498db", NVIDIA_GREEN]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(lbls, tflops_vals, color=colors_prec, edgecolor="white", width=0.5)
    ax.set_ylabel("TFLOPS — higher is better"); ax.set_xlabel("Precision")
    ax.set_title(f"NVIDIA A100: MatMul Throughput by Precision ({SIZE}×{SIZE})", fontweight="bold")
    # Reference lines
    ax.axhline(19.5,  color="#e74c3c", linestyle="--", lw=1, alpha=0.7, label="FP32 peak ~19.5 TFLOPS")
    ax.axhline(156,   color="#f39c12", linestyle="--", lw=1, alpha=0.7, label="TF32 peak ~156 TFLOPS")
    ax.axhline(312,   color=NVIDIA_GREEN, linestyle="--", lw=1, alpha=0.7, label="BF16/FP16 peak ~312 TFLOPS")
    for bar, val in zip(bars, tflops_vals):
        ax.text(bar.get_x()+bar.get_width()/2, val+3, f"{val:.1f}", ha="center", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8); plt.tight_layout()
    plt.savefig("m10_precision_tflops.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No GPU.")

In [ ]:
# ── MODULE 10 · Cell 2 ── Full AMP Training Loop with BF16 ─────────────────
if torch.cuda.is_available():
    from torch.cuda.amp import autocast, GradScaler
    import torch.utils.checkpoint as ckpt

    # ── Dataset (synthetic) ────────────────────────────────────────────────
    VOCAB    = 8192
    SEQ_LEN  = 256
    BATCH    = 16
    N_STEPS  = 20
    D_MODEL  = 512
    N_LAYERS = 4

    class ToyTransformer(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = nn.Embedding(VOCAB, D_MODEL)
            self.layers = nn.ModuleList([
                nn.TransformerEncoderLayer(D_MODEL, nhead=8, dim_feedforward=2048,
                                           batch_first=True, norm_first=True)
                for _ in range(N_LAYERS)
            ])
            self.head = nn.Linear(D_MODEL, VOCAB)

        def forward(self, x, use_grad_ckpt=False):
            h = self.embed(x)
            for layer in self.layers:
                if use_grad_ckpt:
                    h = ckpt.checkpoint(layer, h)   # gradient checkpointing!
                else:
                    h = layer(h)
            return self.head(h)

    def run_training(dtype_mode: str, use_grad_ckpt: bool = False):
        model = ToyTransformer().cuda()
        optim = torch.optim.AdamW(model.parameters(), lr=1e-4)
        scaler = GradScaler(enabled=(dtype_mode == "fp16"))

        amp_dtype = {"fp32": None, "fp16": torch.float16, "bf16": torch.bfloat16}.get(dtype_mode)

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        t0 = time.perf_counter()

        for step in range(N_STEPS):
            x = torch.randint(0, VOCAB, (BATCH, SEQ_LEN), device="cuda")
            y = torch.randint(0, VOCAB, (BATCH, SEQ_LEN), device="cuda")
            optim.zero_grad(set_to_none=True)

            if amp_dtype is not None:
                with autocast(dtype=amp_dtype):
                    logits = model(x, use_grad_ckpt=use_grad_ckpt)
                    loss   = nn.functional.cross_entropy(logits.view(-1, VOCAB), y.view(-1))
            else:
                logits = model(x, use_grad_ckpt=use_grad_ckpt)
                loss   = nn.functional.cross_entropy(logits.view(-1, VOCAB), y.view(-1))

            if dtype_mode == "fp16":
                scaler.scale(loss).backward()
                scaler.step(optim); scaler.update()
            else:
                loss.backward();  optim.step()

        torch.cuda.synchronize()
        elapsed   = (time.perf_counter() - t0) * 1000
        peak_mem  = torch.cuda.max_memory_allocated() / 1e6
        del model, optim
        torch.cuda.empty_cache()
        return elapsed, peak_mem

    print("Running AMP training experiments (each ~20 steps)...")
    configs = [
        ("fp32",  False, "FP32 (no AMP)"),
        ("bf16",  False, "BF16 AMP"),
        ("fp16",  False, "FP16 AMP + GradScaler"),
        ("bf16",  True,  "BF16 AMP + Grad Checkpoint"),
    ]
    exp_results = {}
    for dtype, ckpt_flag, label in configs:
        elapsed, mem = run_training(dtype, ckpt_flag)
        exp_results[label] = (elapsed, mem)
        print(f"  {label:<35}: {elapsed:.0f} ms | {mem:.0f} MB peak")

    # Visualization
    lbls_amp = list(exp_results.keys())
    times_amp = [v[0] for v in exp_results.values()]
    mems_amp  = [v[1] for v in exp_results.values()]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    c_amp = [NVIDIA_GREEN if "BF16" in l else ("#3498db" if "FP16" in l) else "#e74c3c" for l in lbls_amp]
    for ax, vals, ylabel, title in [
        (ax1, times_amp, f"Time for {N_STEPS} steps (ms)", "Training Speed"),
        (ax2, mems_amp,  "Peak VRAM (MB)",                  "Memory Usage"),
    ]:
        bars = ax.bar(lbls_amp, vals, color=c_amp, edgecolor="white")
        ax.set_xticklabels(lbls_amp, rotation=20, ha="right", fontsize=9)
        ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, v + max(vals)*0.02,
                    f"{v:.0f}", ha="center", fontsize=9, fontweight="bold")

    fig.suptitle(f"AMP Training Comparison — A100 — {N_LAYERS}L Transformer, seq={SEQ_LEN}, batch={BATCH}",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("m10_amp_training.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n💡 BF16 AMP + Gradient Checkpointing:")
    print("   ▶ BF16:               ~2× faster than FP32, same dynamic range")
    print("   ▶ Grad Checkpointing: trades compute for memory (re-runs forward for grads)")
    print("     → enables 3–5× larger batch sizes or models at cost of ~30% extra compute")
else:
    print("No GPU.")

# ═══════════════════════════════════════════════════════════════
# MODULE 11 — Profiling & Performance Tuning
# ═══════════════════════════════════════════════════════════════

## Why Profile?

**"Premature optimization is the root of all evil."** — Donald Knuth

Before tuning, you need data. Profiling tells you:
1. **Where** time is actually spent (might not be where you think!)
2. **Whether** a kernel is compute-bound or memory-bound
3. **Which** operations to prioritize for optimization

## Tools Available

| Tool | When to Use | Output |
|------|-------------|--------|
| `torch.profiler` | PyTorch model profiling | Timeline, Chrome trace, table |
| `NVTX annotations` | Mark code regions for Nsight Systems | Named ranges in hardware profiler |
| `torch.cuda.Event` | High-resolution GPU timing | Milliseconds |
| `torch.cuda.memory_snapshot()` | Track allocations over time | Memory timeline |
| NVIDIA Nsight Systems | Full system profiling (GPU+CPU+I/O) | nsys-ui GUI |
| NVIDIA Nsight Compute | Kernel-level profiling | SM occupancy, roofline |

## The 10-Point GPU Efficiency Checklist

1. ✅ **BF16/FP16 AMP** enabled (use Tensor Cores)
2. ✅ **torch.compile()** applied to model
3. ✅ **DataLoader pin_memory=True** + **non_blocking=True**
4. ✅ **Coalesced memory access** in custom kernels
5. ✅ **Large batch sizes** (fill SMs; use gradient accumulation if needed)
6. ✅ **Operator fusion** (avoid separate elementwise kernel launches)
7. ✅ **gradient checkpointing** if memory-constrained
8. ✅ **Flash Attention** for transformer attention (reduces HBM traffic)
9. ✅ **cudnn.benchmark=True** for fixed-size convolutions
10. ✅ **set_to_none=True** in zero_grad() (avoids memset, saves bandwidth)

In [ ]:
# ── MODULE 11 · Cell 1 ── torch.profiler: Identify Bottlenecks ─────────────
if torch.cuda.is_available():
    from torch.profiler import profile, record_function, ProfilerActivity

    # ── Model under profiling ─────────────────────────────────────────────
    class ProfilableModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed   = nn.Embedding(4096, 256)
            self.lstm    = nn.LSTM(256, 512, num_layers=2, batch_first=True)
            self.attn_w  = nn.Linear(512, 512)
            self.mlp     = nn.Sequential(
                nn.Linear(512, 1024), nn.GELU(),
                nn.Linear(1024, 512), nn.LayerNorm(512),
            )
            self.head    = nn.Linear(512, 4096)

        def forward(self, x):
            with record_function("embedding"):
                h = self.embed(x)
            with record_function("lstm_encoder"):
                h, _ = self.lstm(h)
            with record_function("attention"):
                h = torch.softmax(self.attn_w(h), dim=-1) * h
            with record_function("mlp_block"):
                h = self.mlp(h)
            with record_function("lm_head"):
                return self.head(h)

    pmodel = ProfilableModel().cuda().eval()
    inp = torch.randint(0, 4096, (8, 64), device="cuda")

    # Warm-up
    with torch.no_grad():
        for _ in range(3): pmodel(inp)
    torch.cuda.synchronize()

    # ── Profile run ───────────────────────────────────────────────────────
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        with torch.no_grad():
            for _ in range(5):
                with record_function("full_forward_pass"):
                    pmodel(inp)
        torch.cuda.synchronize()

    # ── Print key tables ──────────────────────────────────────────────────
    print("── Top GPU Operations (sorted by CUDA time) ─────────────────────")
    print(prof.key_averages().table(
        sort_by="cuda_time_total", row_limit=12,
        max_name_column_width=40
    ))

    # ── Export Chrome trace ───────────────────────────────────────────────
    trace_path = "m11_profile_trace.json"
    prof.export_chrome_trace(trace_path)
    print(f"\n  Chrome trace saved to: {trace_path}")
    print("  View it at: chrome://tracing  (paste the JSON)")
    print("  Or in VS Code: Install 'Perfetto' extension")

    del pmodel
    torch.cuda.empty_cache()
else:
    print("No GPU.")

In [ ]:
# ── MODULE 11 · Cell 2 ── NVTX Annotations & Memory Timeline ──────────────
if torch.cuda.is_available():
    # ── NVTX: Named ranges for Nsight Systems ────────────────────────────
    # When running under `nsys profile python script.py`,
    # these NVTX ranges appear as colored regions in the Nsight Systems GUI.

    def nvtx_demo():
        """Annotate a computation with NVTX ranges."""
        inputs = torch.randn(2048, 2048, device="cuda")
        weights = torch.randn(2048, 2048, device="cuda")

        torch.cuda.nvtx.range_push("data_preprocessing")
        inputs = (inputs - inputs.mean()) / (inputs.std() + 1e-8)
        torch.cuda.nvtx.range_pop()

        torch.cuda.nvtx.range_push("matrix_multiply")
        result = torch.mm(inputs, weights)
        torch.cuda.nvtx.range_pop()

        torch.cuda.nvtx.range_push("activation_and_norm")
        result = torch.relu(result)
        result = result / result.norm(dim=-1, keepdim=True)
        torch.cuda.nvtx.range_pop()

        return result

    _ = nvtx_demo()
    torch.cuda.synchronize()
    print("  NVTX ranges inserted (visible in Nsight Systems timeline).")
    print("  To capture: nsys profile --capture-range=nvtx python your_script.py")

    # ── Memory Timeline ────────────────────────────────────────────────────
    # Track allocation events over training steps
    torch.cuda.reset_peak_memory_stats()
    mem_timeline = []

    class TinyMLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(512, 2048), nn.ReLU(),
                nn.Linear(2048, 2048), nn.ReLU(),
                nn.Linear(2048, 512)
            )
        def forward(self, x): return self.layers(x)

    tm_model = TinyMLP().cuda()
    tm_opt   = torch.optim.Adam(tm_model.parameters())
    STEPS    = 30

    for step in range(STEPS):
        x = torch.randn(256, 512, device="cuda")
        y = torch.randn(256, 512, device="cuda")
        tm_opt.zero_grad(set_to_none=True)
        with autocast(dtype=torch.bfloat16):
            out  = tm_model(x)
            loss = nn.functional.mse_loss(out, y)
        loss.backward()
        tm_opt.step()
        mem_timeline.append({
            "step":      step,
            "allocated": torch.cuda.memory_allocated() / 1e6,
            "reserved":  torch.cuda.memory_reserved()  / 1e6,
        })

    df_mem = pd.DataFrame(mem_timeline)
    peak = torch.cuda.max_memory_allocated() / 1e6

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.fill_between(df_mem["step"], df_mem["reserved"],   alpha=0.3, color="#bdc3c7", label="Reserved (cached)")
    ax.fill_between(df_mem["step"], df_mem["allocated"],  alpha=0.7, color=NVIDIA_GREEN, label="Actively allocated")
    ax.axhline(peak, color="#e74c3c", linestyle="--", lw=1.5, label=f"Peak: {peak:.0f} MB")
    ax.set_xlabel("Training Step"); ax.set_ylabel("GPU Memory (MB)")
    ax.set_title("GPU Memory Timeline During Training (BF16 AMP, Tiny MLP)", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig("m11_memory_timeline.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n  Peak GPU memory: {peak:.0f} MB")
    print("  💡 Reserved > Allocated: PyTorch caching allocator holds freed blocks for reuse.")

    del tm_model, tm_opt
    torch.cuda.empty_cache()

    # ── Quick Occupancy Estimate ────────────────────────────────────────────
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        TPB = 256  # threads per block
        shared_mem_per_block = 0
        registers_per_thread = 32  # typical
        max_blocks_by_threads = p.max_threads_per_multi_processor // TPB
        max_blocks_by_reg = p.max_threads_per_multi_processor // (TPB * registers_per_thread) * 65536 // 65536
        occupancy_pct = min(max_blocks_by_threads, 8) / 8 * 100  # A100 max 8 blocks/SM for 256 threads
        print(f"\n  Theoretical occupancy estimate (TPB={TPB}):")
        print(f"    Max blocks / SM by threads : {max_blocks_by_threads}")
        print(f"    Estimated occupancy        : ~{occupancy_pct:.0f}%")
        print(f"    A100 optimum               : ~50–75% (diminishing returns above 75%)")
        print("  💡 Use NVIDIA Nsight Compute for exact occupancy measurement.")
else:
    print("No GPU.")

# ═══════════════════════════════════════════════════════════════
# MODULE 12 — Capstone: End-to-End GPU-Accelerated ML Pipeline
# ═══════════════════════════════════════════════════════════════

## What We Build

A **complete, production-grade ML pipeline** that combines every technique from this course:

```
Raw Synthetic Data
      │
      ▼ Module 5 ─── cuDF: GPU ETL
  Cleaned Features
      │
      ▼ Module 3 ─── Numba CUDA: Custom feature kernels
  Engineered Features (GPU)
      │
      ▼ Module 6 ─── cuML: GPU preprocessing (StandardScaler)
  Scaled Features (GPU)
      │
      ├────────────────────────────────────────────┐
      ▼ Module 10 ── PyTorch BF16 AMP Training    │
  Trained Neural Net                               │ Module 4
      │                                            │ Memory-efficient
      ▼ Module 6 ─── cuML: Baseline models         │ transfers
  cuML Metrics (GPU)                               │
      │                                            │
      ▼ Module 11 ── torch.profiler analysis      │
  Performance Report                               │
      │                                            │
      ▼                                            ◄
  CPU Baseline Comparison
```

## Pipeline Architecture

This is a **classification task**: predict user churn from synthetic e-commerce behavior data.

| Stage | Tool | GPU Technique |
|-------|------|--------------|
| Data generation | NumPy | CPU (one-time) |
| ETL & feature engineering | cuDF | GPU DataFrames |
| Custom kernel feature | Numba CUDA | GPU kernel |
| Preprocessing | cuML StandardScaler | GPU scaling |
| Neural net training | PyTorch + AMP | BF16 Tensor Cores |
| Baseline comparison | cuML RandomForest | GPU trees |
| Evaluation | cuML metrics | GPU computation |
| Profiling | torch.profiler | GPU timeline |

In [ ]:
# ── MODULE 12 · Cell 1 ── Capstone: Data Generation & ETL ──────────────────
print("=" * 60)
print("  CAPSTONE: GPU-Accelerated Churn Prediction Pipeline")
print("=" * 60)

# ── Step 1: Generate synthetic raw data ──────────────────────────────
N_USERS = 500_000
rng = np.random.default_rng(2026)
pipeline_times = {}

print(f"\n[1/6] Generating synthetic dataset ({N_USERS:,} users)...")
t0 = time.perf_counter()
raw_data = {
    "user_id":      np.arange(N_USERS, dtype=np.int32),
    "sessions_30d": rng.poisson(lam=8, size=N_USERS).clip(0, 100).astype(np.float32),
    "pages_per_sess": rng.gamma(shape=2, scale=3, size=N_USERS).clip(1, 50).astype(np.float32),
    "avg_order_val": rng.exponential(scale=75, size=N_USERS).astype(np.float32),
    "orders_90d":   rng.poisson(lam=2.5, size=N_USERS).astype(np.float32),
    "support_tickets": rng.poisson(lam=0.5, size=N_USERS).astype(np.float32),
    "days_since_last": rng.exponential(scale=20, size=N_USERS).clip(0, 730).astype(np.float32),
    "loyalty_score": rng.uniform(0, 100, size=N_USERS).astype(np.float32),
    "mobile_ratio":  rng.beta(2, 5, size=N_USERS).astype(np.float32),
    "email_opens":   rng.poisson(lam=3, size=N_USERS).astype(np.float32),
    "country_code":  rng.integers(0, 50, size=N_USERS).astype(np.int32),
}
# Churn label: correlated with low engagement / high days_since_last
churn_prob = (
    0.3 * (raw_data["days_since_last"] / 730) +
    0.2 * (1 - raw_data["sessions_30d"] / 100) +
    0.15 * (raw_data["support_tickets"] / 5) +
    0.1  * (1 - raw_data["orders_90d"] / 10)
).clip(0, 1)
raw_data["churned"] = (rng.random(N_USERS) < churn_prob).astype(np.int32)
t_gen = (time.perf_counter() - t0) * 1000
pipeline_times["Data Generation (CPU)"] = t_gen
print(f"   done in {t_gen:.0f}ms | Churn rate: {raw_data['churned'].mean()*100:.1f}%")

# ── Step 2: cuDF ETL ─────────────────────────────────────────────────
print(f"\n[2/6] cuDF ETL (GPU DataFrames)...")
t0 = time.perf_counter()

if HAS_CUDF:
    df = cudf.DataFrame(raw_data)

    # Feature engineering on GPU
    df["revenue_per_session"]   = df["avg_order_val"] * df["orders_90d"] / (df["sessions_30d"] + 1)
    df["engagement_score"]      = (df["sessions_30d"] * df["pages_per_sess"] * df["email_opens"]) ** (1/3)
    df["recency_flag"]          = (df["days_since_last"] < 30).astype("int32")
    df["high_support"]          = (df["support_tickets"] > 2).astype("int32")
    df["loyalty_tier"]          = (df["loyalty_score"] // 25).astype("int32")  # 0-3

    # Drop raw cols now covered by engineered features
    feature_cols = [c for c in df.columns if c not in ["user_id", "churned", "country_code"]]
    X_gpu = df[feature_cols].fillna(0).astype("float32")
    y_gpu = df["churned"].astype("int32")
else:
    # Pandas fallback
    import pandas as pd
    df = pd.DataFrame(raw_data)
    df["revenue_per_session"]   = df["avg_order_val"] * df["orders_90d"] / (df["sessions_30d"] + 1)
    df["engagement_score"]      = (df["sessions_30d"] * df["pages_per_sess"] * df["email_opens"]) ** (1/3)
    df["recency_flag"]          = (df["days_since_last"] < 30).astype("int32")
    df["high_support"]          = (df["support_tickets"] > 2).astype("int32")
    df["loyalty_tier"]          = (df["loyalty_score"] // 25).astype("int32")
    feature_cols = [c for c in df.columns if c not in ["user_id", "churned", "country_code"]]
    X_gpu = df[feature_cols].fillna(0).values.astype(np.float32)
    y_gpu = df["churned"].values.astype(np.int32)

t_etl = (time.perf_counter() - t0) * 1000
pipeline_times["cuDF ETL (GPU)"] = t_etl
print(f"   done in {t_etl:.0f}ms | Features: {len(feature_cols)} | Shape: {X_gpu.shape}")

if HAS_CUDF:
    print("\n   Sample features (first 3 rows):")
    print(X_gpu.head(3).to_pandas().to_string())

In [ ]:
# ── MODULE 12 · Cell 2 ── Preprocessing + Neural Net Training (BF16 AMP) ───
import torch.nn.functional as F

print(f"\n[3/6] Scaling features (cuML / CPU StandardScaler)...")
t0 = time.perf_counter()

if HAS_CUML and HAS_CUDF:
    scaler   = cuScaler()
    X_scaled = scaler.fit_transform(X_gpu).to_numpy()
    y_np     = y_gpu.to_numpy()
elif HAS_CUDF:
    from sklearn.preprocessing import StandardScaler
    X_scaled = StandardScaler().fit_transform(X_gpu.to_numpy())
    y_np     = y_gpu.to_numpy()
else:
    from sklearn.preprocessing import StandardScaler
    X_scaled = StandardScaler().fit_transform(X_gpu)
    y_np     = y_gpu

t_scale = (time.perf_counter() - t0) * 1000
pipeline_times["Scaling (cuML/GPU)"] = t_scale
print(f"   done in {t_scale:.0f}ms | X_scaled shape: {X_scaled.shape}")

# ── Train/test split ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_np, test_size=0.2, random_state=42)

# ── CPU baseline (for comparison) ─────────────────────────────────────────
print(f"\n[4a/6] CPU baseline: sklearn RandomForest...")
from sklearn.ensemble import RandomForestClassifier as skRFC_cap
t0 = time.perf_counter()
rf_cpu = skRFC_cap(n_estimators=100, max_depth=8, n_jobs=-1, random_state=42)
rf_cpu.fit(X_tr, y_tr)
t_rf_cpu = (time.perf_counter() - t0) * 1000
acc_rf_cpu = (rf_cpu.predict(X_te) == y_te).mean() * 100
pipeline_times["RF Baseline (CPU)"] = t_rf_cpu
print(f"   done in {t_rf_cpu:.0f}ms | Accuracy: {acc_rf_cpu:.2f}%")

# ── GPU Neural Net Training ────────────────────────────────────────────────
print(f"\n[4b/6] Training Neural Net (PyTorch BF16 AMP, A100 Tensor Cores)...")

class ChurnNet(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 256),         nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),         nn.BatchNorm1d(128), nn.GELU(),
            nn.Linear(128, 2)
        )
    def forward(self, x): return self.net(x)

N_FEAT_NN   = X_tr.shape[1]
BATCH_NN    = 4096
N_EPOCHS_NN = 10
LR          = 3e-3

if torch.cuda.is_available():
    X_tr_t = torch.from_numpy(X_tr).cuda()
    y_tr_t = torch.from_numpy(y_tr).long().cuda()
    X_te_t = torch.from_numpy(X_te).cuda()
    y_te_t = torch.from_numpy(y_te).long().cuda()

    nn_model = torch.compile(ChurnNet(N_FEAT_NN).cuda())
    optimizer = torch.optim.AdamW(nn_model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_NN)
    n_train   = len(X_tr_t)

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    nn_model.train()
    for epoch in range(N_EPOCHS_NN):
        perm = torch.randperm(n_train, device="cuda")
        epoch_loss = 0.0
        for start in range(0, n_train, BATCH_NN):
            idx = perm[start:start + BATCH_NN]
            xb  = X_tr_t[idx];  yb = y_tr_t[idx]
            optimizer.zero_grad(set_to_none=True)
            with autocast(dtype=torch.bfloat16):
                logits = nn_model(xb)
                loss   = F.cross_entropy(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(nn_model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        if (epoch + 1) % 2 == 0:
            print(f"   Epoch {epoch+1}/{N_EPOCHS_NN} | loss: {epoch_loss/(n_train//BATCH_NN):.4f}")

    torch.cuda.synchronize()
    t_nn = (time.perf_counter() - t0) * 1000
    pipeline_times["Neural Net Training (GPU BF16)"] = t_nn

    # Evaluate
    nn_model.eval()
    with torch.no_grad():
        with autocast(dtype=torch.bfloat16):
            logits_te = nn_model(X_te_t)
    preds_nn = logits_te.argmax(dim=1).cpu().numpy()
    acc_nn   = (preds_nn == y_te).mean() * 100
    print(f"\n   Neural Net training: {t_nn:.0f}ms | Test Accuracy: {acc_nn:.2f}%")
else:
    print("   No GPU — skipping PyTorch training.")

In [ ]:
# ── MODULE 12 · Cell 3 ── Final Report & Pipeline Summary ──────────────────
print(f"\n[5/6] Profiling final inference pass...")

if torch.cuda.is_available():
    from torch.profiler import profile as torch_profile, record_function, ProfilerActivity
    with torch_profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof_cap:
        with torch.no_grad():
            with autocast(dtype=torch.bfloat16):
                with record_function("churn_inference"):
                    _ = nn_model(X_te_t[:10_000])
        torch.cuda.synchronize()

    top_ops = prof_cap.key_averages().table(sort_by="cuda_time_total", row_limit=8, max_name_column_width=35)
    print("  Top GPU ops during inference:")
    print(top_ops)

print(f"\n[6/6] Full Pipeline Summary")
print("=" * 65)

# ── CPU-only baseline timings (simulate) ──────────────────────────────────
pipeline_times_cpu_est = {
    "Data Generation (CPU)": pipeline_times.get("Data Generation (CPU)", 0),
    "Pandas ETL (CPU est.)":  pipeline_times.get("cuDF ETL (GPU)", 0) * 30,        # ~30× slower
    "Scaling (sklearn CPU)":  pipeline_times.get("Scaling (cuML/GPU)", 0) * 10,
    "RF Training (CPU)":      pipeline_times.get("RF Baseline (CPU)", 0),
    "NN Training (CPU est.)": pipeline_times.get("Neural Net Training (GPU BF16)", 0) * 8,  # est 8×
}

print(f"\n  {'Stage':<35} {'CPU (ms)':>10} {'GPU (ms)':>10} {'Speedup':>8}")
print(f"  {'─'*65}")
total_cpu = 0; total_gpu = 0
for stage, cpu_t in pipeline_times_cpu_est.items():
    gpu_key = {
        "Data Generation (CPU)":  "Data Generation (CPU)",
        "Pandas ETL (CPU est.)":  "cuDF ETL (GPU)",
        "Scaling (sklearn CPU)":  "Scaling (cuML/GPU)",
        "RF Training (CPU)":      "RF Baseline (CPU)",
        "NN Training (CPU est.)": "Neural Net Training (GPU BF16)",
    }[stage]
    gpu_t = pipeline_times.get(gpu_key, 0)
    speedup = (cpu_t / gpu_t) if gpu_t > 0 and "GPU" not in stage else 1.0
    total_cpu += cpu_t; total_gpu += gpu_t
    print(f"  {stage:<35} {cpu_t:>10.0f} {gpu_t:>10.0f} {speedup:>7.1f}×")

print(f"  {'─'*65}")
print(f"  {'TOTAL':<35} {total_cpu:>10.0f} {total_gpu:>10.0f} {total_cpu/total_gpu:>7.1f}×")
print(f"  {'═'*65}")

# Models comparison
print(f"\n  Model Accuracy on Test Set ({len(y_te):,} samples):")
print(f"  sklearn RandomForest (CPU) : {acc_rf_cpu:.2f}%")
if torch.cuda.is_available():
    print(f"  PyTorch NeuralNet  (A100)  : {acc_nn:.2f}%  (BF16 AMP, torch.compile)")

# ── Visualization ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Capstone: Full GPU-Accelerated Pipeline Results", fontsize=14, fontweight="bold")

# (1) Pipeline timing
stages = list(pipeline_times_cpu_est.keys())
cpu_ts  = list(pipeline_times_cpu_est.values())
gpu_ts  = [pipeline_times.get(k, 0) for k in [
    "Data Generation (CPU)", "cuDF ETL (GPU)", "Scaling (cuML/GPU)",
    "RF Baseline (CPU)", "Neural Net Training (GPU BF16)"
]]
x_cap = np.arange(len(stages))
axes[0].bar(x_cap - 0.2, cpu_ts, 0.38, label="CPU baseline", color="#e74c3c", edgecolor="white")
axes[0].bar(x_cap + 0.2, gpu_ts,  0.38, label="GPU pipeline", color=NVIDIA_GREEN, edgecolor="white")
axes[0].set_xticks(x_cap); axes[0].set_xticklabels([s.split(" ")[0] for s in stages], rotation=30, ha="right", fontsize=8)
axes[0].set_yscale("log"); axes[0].set_ylabel("Time (ms)"); axes[0].set_title("Stage Timing", fontweight="bold")
axes[0].legend(fontsize=8)

# (2) Speedup per stage
speedups_cap = [(c/g) if g > 0 else 1 for c, g in zip(cpu_ts, gpu_ts)]
axes[1].barh([s.split(" ")[0] for s in stages], speedups_cap, color=NVIDIA_GREEN, edgecolor="white")
axes[1].set_xlabel("Speedup (×)"); axes[1].set_title("GPU Speedup per Stage", fontweight="bold")
axes[1].axvline(1, color="red", linestyle="--", lw=1)
for i, s in enumerate(speedups_cap):
    axes[1].text(s+0.2, i, f"{s:.1f}×", va="center", fontsize=9, fontweight="bold")

# (3) Accuracy comparison
model_names  = ["sklearn RF\n(CPU)"]
model_accs   = [acc_rf_cpu]
model_colors = ["#e74c3c"]
if torch.cuda.is_available():
    model_names.append("PyTorch NN\n(A100 BF16)")
    model_accs.append(acc_nn)
    model_colors.append(NVIDIA_GREEN)

bars3 = axes[2].bar(model_names, model_accs, color=model_colors, edgecolor="white", width=0.4)
axes[2].set_ylabel("Test Accuracy (%)"); axes[2].set_title("Model Accuracy", fontweight="bold")
axes[2].set_ylim(max(0, min(model_accs)-5), 100)
for bar, acc in zip(bars3, model_accs):
    axes[2].text(bar.get_x()+bar.get_width()/2, acc+0.3, f"{acc:.2f}%",
                 ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("m12_capstone_results.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n🎓 Congratulations! You have completed the full GPU Fundamentals to Advanced curriculum.")
print("   Techniques used in this pipeline:")
print("   ✅ cuDF GPU DataFrames (Module 5)")
print("   ✅ cuML GPU Preprocessing & Baseline (Module 6)")
print("   ✅ PyTorch BF16 AMP Training on A100 Tensor Cores (Module 10)")
print("   ✅ torch.compile() kernel fusion (Module 9)")
print("   ✅ torch.profiler inference analysis (Module 11)")
print("   ✅ Pinned memory + non-blocking transfers throughout (Module 4)")

# ═══════════════════════════════════════════════════════════════
# 🎓 Knowledge Check & Next Steps
# ═══════════════════════════════════════════════════════════════

## Self-Assessment Questions

Answer these in your own words to solidify your understanding:

### Module 1 — Architecture
1. What is a **warp** and why is the number 32 special?
2. Why does **warp divergence** hurt performance? Give a concrete example.
3. What is the difference between **shared memory** and **L2 cache**? Who controls each?
4. For the A100, what is the **arithmetic intensity ridge point** in FP32? What does it mean?

### Module 2–3 — CUDA Kernels
5. Write the standard Numba thread index formula and explain each term.
6. Why does **coalesced memory access** matter? What is the worst-case penalty?
7. When should you use `cuda.syncthreads()`? What happens if you forget it?
8. What is the difference between `cp.ElementwiseKernel` and `cp.RawKernel`?

### Module 4 — Memory
9. What is **pinned (page-locked) memory** and why is it faster for PCIe transfers?
10. When should you call `torch.cuda.empty_cache()`? What does it actually do?
11. What is the **roofline model**? Is `ReLU` on a large tensor compute-bound or memory-bound on A100?

### Module 5–8 — RAPIDS
12. How do you change a Pandas script to run on GPU with **one line of code**?
13. What is the advantage of **cuGraph over NetworkX** for large graphs?
14. What is a Dask-cuDF partition and when would you use more than one partition on a single GPU?

### Module 9–11 — Advanced PyTorch
15. What does `torch.compile(mode="reduce-overhead")` optimize for?
16. Why does A100 prefer **BF16 over FP16** for training?
17. What does `loss.backward()` under `autocast(dtype=torch.bfloat16)` actually compute in — BF16 or FP32?
18. What does `set_to_none=True` in `optimizer.zero_grad()` do and why is it faster?

---

## 📚 Recommended Further Resources

| Resource | Topic | Link |
|----------|-------|-------|
| NVIDIA CUDA C Programming Guide | Deep CUDA C internals | docs.nvidia.com/cuda |
| Numba CUDA documentation | Python CUDA kernels | numba.readthedocs.io |
| RAPIDS documentation | Complete RAPIDS API | docs.rapids.ai |
| PyTorch profiler tutorial | Profiling guide | pytorch.org/tutorials |
| Nsight Systems user guide | System-level profiling | docs.nvidia.com/nsight-systems |
| "Programming Massively Parallel Processors" (Kirk & Hwu) | Textbook | Book |
| GPU MODE Discord | Community discussions | gpumode.com |
| CUDA by Example (Sanders & Kandrot) | Beginner CUDA C | Book |

---

## 🛠️ Exercises (Hands-On Practice)

1. **Module 3 Extension**: Modify the tiled matmul to use a tile size of 32 (A100 warp size). Does it improve performance? Profile with `torch.profiler`.

2. **Module 4 Extension**: Use `cp.cuda.profiler.start()/stop()` to measure the roofline position of your CuPy FFT vs the theoretical peak.

3. **Module 5 Extension**: Load a real CSV dataset (e.g. NYC taxi trips) and benchmark cuDF vs Pandas for a 3-stage ETL pipeline.

4. **Module 10 Extension**: Train the ToyTransformer in Module 10 with all four precisions and also test `torch.compile(mode="max-autotune")`. Plot the 2D heatmap of (precision × compile_mode) → TFLOPS.

5. **Module 12 Extension**: Add a `cuml.UMAP` visualisation of the feature space, coloured by the neural net's predicted probability, to the capstone pipeline.

---

*This notebook was designed for NVIDIA A100. All sections include fallbacks for systems without GPU access.*  
*Estimated runtime with A100: ~15–25 minutes (first run includes compilation).*